# B8 · RL moderne : DQN, gradient de politique, PPO et RLHF — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b8_rl_moderne/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b8_rl_moderne/03_notebook.ipynb`, créée par `python tools/start_chapter.py B8`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B8.Q1 à B8.Q12, B8.R1 à B8.R3, B8.1 à B8.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B8.12 | CartPole pour le deep RL : espaces, politique aléatoire, evaluate_policy et la référence tabulaire du ch. 26 | 📦 | ★ | 15 |
|  | B8.13 | DQN sur CartPole : réseau Q, ReplayBuffer de rl.py, réseau cible, Huber, ε décroissant | 📦 | ★★★ | 60 |
|  | B8.14 | Le DQN qui diverge : quatre erreurs de cible, de fin d'épisode et d'exploration | 🐛 | ★★ | 30 |
|  | B8.15 | Ablations du DQN : sans réseau cible, sans rejeu, avec Double DQN — trois graines 🚀 | 🔬 | ★★★ | 50 |
| B | B8.16 | Prédire : REINFORCE sans baseline, avec baseline, avec retours normalisés — lequel apprend le plus vite ? | 🔮 | ★ | 10 |
|  | B8.17 | REINFORCE avec et sans baseline : variance du gradient et vitesse d'apprentissage | 📦 | ★★★ | 45 |
|  | B8.18 | compute_gae et explained_variance | 🔨 | ★★ | 30 |
|  | B8.19 | clipped_surrogate : valeur et gradient par rapport au ratio | 🔨 | ★★ | 20 |
|  | B8.20 | PPO from scratch sur CartPole : environnements vectorisés, acteur-critique, GAE, clip, entropie | 📦 | ★★★★ | 150 |
| B | B8.21 à B8.24 | la fin de la politique : les journaux de ton PPO, ses hyperparamètres sur trois graines, sa robustesse face à une autre physique (*prochaine session de génération*) | | | |
| C | B8.25 à B8.28 | les préférences : Bradley-Terry et DPO dans ta librairie, un RLHF miniature sur CartPole, la revue de code face à CleanRL, et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B8")  # your own library

## Objectifs et rappel express

- Expliquer pourquoi et comment un réseau remplace la table Q, et ce qui rend DQN stable.
- Dériver le gradient de politique et justifier la baseline, l'avantage et GAE.
- Implémenter en PyTorch un DQN, REINFORCE et PPO sur CartPole, avec les calculs clés testés dans `mylearn`.
- Diagnostiquer un entraînement RL à partir de ses journaux et comparer des variantes sur plusieurs graines.
- Expliquer RLHF, DPO et GRPO et entraîner un modèle de récompense sur des préférences.

**Rappel express.** Cible DQN $y = r + \gamma(1 - d)\max_{a'} Q_{\bar\theta}(s', a')$ ; Double DQN : $Q_{\bar\theta}(s', \arg\max_{a'} Q_\theta(s', a'))$. Huber : $\frac12 x^2$ si $|x| \le \delta$, $\delta(|x| - \frac{\delta}{2})$ sinon. $\nabla_{\mathbf z} \log \operatorname{softmax}(\mathbf z)_a = \mathbf e_a - \operatorname{softmax}(\mathbf z)$. $\delta_t = r_t + \gamma(1 - \text{terminated}_t)V(s_{t+1}) - V(s_t)$, $\hat A_t = \delta_t + \gamma\lambda(1 - \text{done}_t)\hat A_{t+1}$. PPO : $-\operatorname{mean}\min(\rho \hat A, \operatorname{clip}(\rho, 1 \pm \varepsilon)\hat A)$. Bradley-Terry : $-\log\sigma(r_w - r_l)$ ; DPO : $-\log\sigma\big(\beta[(\log\pi_w - \log\pi^{\text{ref}}_w) - (\log\pi_l - \log\pi^{\text{ref}}_l)]\big)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B8.1 à B8.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`) ; une forme : entre guillemets (`"(3, 5)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B8.7, l'estimation 🧮 B8.8, la réflexion (🗣️ B8.9, ⚖️ B8.10, 📄 B8.11) et l'entretien se corrigent avec `05_solutions.md`.

**B8.Q1 — Pourquoi une table Q ne suffit plus pour CartPole**

In [ ]:
# B8.Q1: the values you found on paper
answer_B8_Q1a = ...  # a) a whole number
answer_B8_Q1b = ...  # b) a whole number
answer_B8_Q1c = ...  # c) True or False
answer_B8_Q1d = ...  # d) True or False
answer_B8_Q1e = ...  # e) a shape as a string, e.g. "(3, 5)"

for letter, answer in zip("abcde", [answer_B8_Q1a, answer_B8_Q1b, answer_B8_Q1c, answer_B8_Q1d, answer_B8_Q1e]):
    wb.check(f"B8.Q1{letter}", answer)

**B8.Q2 — La triade mortelle : approximation, amorçage et apprentissage off-policy**

In [ ]:
# B8.Q2: the values you found on paper
answer_B8_Q2a = ...  # a) the letters, e.g. "EF"
answer_B8_Q2b = ...  # b) a whole number
answer_B8_Q2c = ...  # c) a whole number
answer_B8_Q2d = ...  # d) a whole number
answer_B8_Q2e = ...  # e) True or False
answer_B8_Q2f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q2a, answer_B8_Q2b, answer_B8_Q2c, answer_B8_Q2d, answer_B8_Q2e, answer_B8_Q2f]):
    wb.check(f"B8.Q2{letter}", answer)

**B8.Q3 — Réseau cible et mémoire de rejeu : quel problème chacun résout-il ?**

In [ ]:
# B8.Q3: the values you found on paper
answer_B8_Q3a = ...  # a) a whole number
answer_B8_Q3b = ...  # b) True or False
answer_B8_Q3c = ...  # c) True or False
answer_B8_Q3d = ...  # d) a number, 2 decimals
answer_B8_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q3a, answer_B8_Q3b, answer_B8_Q3c, answer_B8_Q3d, answer_B8_Q3e]):
    wb.check(f"B8.Q3{letter}", answer)

**B8.Q4 — Double DQN : d'où vient la surestimation des valeurs Q ?**

In [ ]:
# B8.Q4: the values you found on paper
answer_B8_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q4b = ...  # b) a number, 2 decimals
answer_B8_Q4c = ...  # c) a whole number
answer_B8_Q4d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q4e = ...  # e) True or False
answer_B8_Q4f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q4a, answer_B8_Q4b, answer_B8_Q4c, answer_B8_Q4d, answer_B8_Q4e, answer_B8_Q4f]):
    wb.check(f"B8.Q4{letter}", answer)

**B8.Q5 — Méthodes par valeur ou par politique : actions continues et politiques aléatoires**

In [ ]:
# B8.Q5: the values you found on paper
answer_B8_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q5b = ...  # b) a whole number
answer_B8_Q5c = ...  # c) a number, 2 decimals
answer_B8_Q5d = ...  # d) the letters, e.g. "EF"
answer_B8_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q5a, answer_B8_Q5b, answer_B8_Q5c, answer_B8_Q5d, answer_B8_Q5e]):
    wb.check(f"B8.Q5{letter}", answer)

**B8.Q6 — REINFORCE : pourquoi soustraire une baseline ne biaise pas le gradient**

In [ ]:
# B8.Q6: the values you found on paper
answer_B8_Q6a = ...  # a) a number, 2 decimals
answer_B8_Q6b = ...  # b) a number, 2 decimals
answer_B8_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_B8_Q6d = ...  # d) True or False
answer_B8_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q6a, answer_B8_Q6b, answer_B8_Q6c, answer_B8_Q6d, answer_B8_Q6e]):
    wb.check(f"B8.Q6{letter}", answer)

**B8.Q7 — Avantage, erreur TD et GAE : ce que règle λ**

In [ ]:
# B8.Q7: the values you found on paper
answer_B8_Q7a = ...  # a) a number, 2 decimals
answer_B8_Q7b = ...  # b) the letter of your choice, e.g. "E"
answer_B8_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_B8_Q7d = ...  # d) a number, 2 decimals
answer_B8_Q7e = ...  # e) True or False
answer_B8_Q7f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q7a, answer_B8_Q7b, answer_B8_Q7c, answer_B8_Q7d, answer_B8_Q7e, answer_B8_Q7f]):
    wb.check(f"B8.Q7{letter}", answer)

**B8.Q8 — PPO : à quoi sert le clip, et pourquoi réutiliser plusieurs fois le même lot ?**

In [ ]:
# B8.Q8: the values you found on paper
answer_B8_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q8b = ...  # b) True or False
answer_B8_Q8c = ...  # c) a whole number
answer_B8_Q8d = ...  # d) a number, 2 decimals
answer_B8_Q8e = ...  # e) True or False
answer_B8_Q8f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q8a, answer_B8_Q8b, answer_B8_Q8c, answer_B8_Q8d, answer_B8_Q8e, answer_B8_Q8f]):
    wb.check(f"B8.Q8{letter}", answer)

**B8.Q9 — On-policy ou off-policy : DQN, REINFORCE et PPO**

In [ ]:
# B8.Q9: the values you found on paper
answer_B8_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_B8_Q9b = ...  # b) True or False
answer_B8_Q9c = ...  # c) True or False
answer_B8_Q9d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B8_Q9a, answer_B8_Q9b, answer_B8_Q9c, answer_B8_Q9d, answer_B8_Q9e]):
    wb.check(f"B8.Q9{letter}", answer)

**B8.Q10 — Évaluer un agent : graines, épisodes, politique déterministe**

In [ ]:
# B8.Q10: the values you found on paper
answer_B8_Q10a = ...  # a) a whole number
answer_B8_Q10b = ...  # b) a number, 1 decimal
answer_B8_Q10c = ...  # c) True or False
answer_B8_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q10e = ...  # e) True or False
answer_B8_Q10f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B8_Q10a, answer_B8_Q10b, answer_B8_Q10c, answer_B8_Q10d, answer_B8_Q10e, answer_B8_Q10f]):
    wb.check(f"B8.Q10{letter}", answer)

**B8.Q11 — RLHF, DPO, GRPO : qui a besoin d'un modèle de récompense ou d'un critique ?**

In [ ]:
# B8.Q11: the values you found on paper
answer_B8_Q11a = ...  # a) the letters, e.g. "EF"
answer_B8_Q11b = ...  # b) the letters, e.g. "EF"
answer_B8_Q11c = ...  # c) a number, 2 decimals
answer_B8_Q11d = ...  # d) True or False
answer_B8_Q11e = ...  # e) the letter of your choice, e.g. "E"
answer_B8_Q11f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_Q11a, answer_B8_Q11b, answer_B8_Q11c, answer_B8_Q11d, answer_B8_Q11e, answer_B8_Q11f]):
    wb.check(f"B8.Q11{letter}", answer)

**B8.Q12 — Piratage de la récompense (reward hacking) : quand l'agent optimise la lettre et pas l'esprit**

In [ ]:
# B8.Q12: the values you found on paper
answer_B8_Q12a = ...  # a) the letters, e.g. "EF"
answer_B8_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_B8_Q12c = ...  # c) True or False
answer_B8_Q12d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_Q12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_Q12a, answer_B8_Q12b, answer_B8_Q12c, answer_B8_Q12d, answer_B8_Q12e]):
    wb.check(f"B8.Q12{letter}", answer)

**B8.R1 — La mise à jour du Q-learning et le rôle de la mémoire de rejeu**

In [ ]:
# B8.R1: the values you found on paper
answer_B8_R1a = ...  # a) a number, 2 decimals
answer_B8_R1b = ...  # b) a number, 3 decimals
answer_B8_R1c = ...  # c) a number, 2 decimals
answer_B8_R1d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_R1e = ...  # e) True or False
answer_B8_R1f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B8_R1a, answer_B8_R1b, answer_B8_R1c, answer_B8_R1d, answer_B8_R1e, answer_B8_R1f]):
    wb.check(f"B8.R1{letter}", answer)

**B8.R2 — Adam et le gradient clipping**

In [ ]:
# B8.R2: the values you found on paper
answer_B8_R2a = ...  # a) a number, 4 decimals
answer_B8_R2b = ...  # b) a number, 3 decimals
answer_B8_R2c = ...  # c) True or False
answer_B8_R2d = ...  # d) the letter of your choice, e.g. "E"
answer_B8_R2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B8_R2a, answer_B8_R2b, answer_B8_R2c, answer_B8_R2d, answer_B8_R2e]):
    wb.check(f"B8.R2{letter}", answer)

**B8.R3 — Exploration, softmax et bandits**

In [ ]:
# B8.R3: the values you found on paper
answer_B8_R3a = ...  # a) a number, 2 decimals
answer_B8_R3b = ...  # b) a number, 3 decimals
answer_B8_R3c = ...  # c) a number, 1 decimal
answer_B8_R3d = ...  # d) True or False
answer_B8_R3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B8_R3a, answer_B8_R3b, answer_B8_R3c, answer_B8_R3d, answer_B8_R3e]):
    wb.check(f"B8.R3{letter}", answer)

**Ex B8.1 — Cible DQN à la main : état terminal, puis cible Double DQN**

In [ ]:
# Ex B8.1: the values you found on paper
answer_B8_1a = ...  # a) a number, 2 decimals
answer_B8_1b = ...  # b) a number, 2 decimals
answer_B8_1c = ...  # c) a number, 2 decimals
answer_B8_1d = ...  # d) a number, 2 decimals
answer_B8_1e = ...  # e) a number, 2 decimals
answer_B8_1f = ...  # f) a number, 2 decimals
answer_B8_1g = ...  # g) True or False
answer_B8_1h = ...  # h) a number, 2 decimals

for letter, answer in zip("abcdefgh", [answer_B8_1a, answer_B8_1b, answer_B8_1c, answer_B8_1d, answer_B8_1e, answer_B8_1f, answer_B8_1g, answer_B8_1h]):
    wb.check(f"B8.1{letter}", answer)

**Ex B8.2 — Loss de Huber et son gradient pour trois erreurs TD**

In [ ]:
# Ex B8.2: the values you found on paper
answer_B8_2a = ...  # a) a number, 2 decimals
answer_B8_2b = ...  # b) a number, 3 decimals
answer_B8_2c = ...  # c) a number, 2 decimals
answer_B8_2d = ...  # d) a number, 1 decimal
answer_B8_2e = ...  # e) a number, 2 decimals
answer_B8_2f = ...  # f) a number, 3 decimals
answer_B8_2g = ...  # g) a number, 3 decimals
answer_B8_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B8_2a, answer_B8_2b, answer_B8_2c, answer_B8_2d, answer_B8_2e, answer_B8_2f, answer_B8_2g, answer_B8_2h]):
    wb.check(f"B8.2{letter}", answer)

**Ex B8.3 — Gradient du log d'une politique softmax à trois actions**

In [ ]:
# Ex B8.3: the values you found on paper
answer_B8_3a = ...  # a) a number, 4 decimals
answer_B8_3b = ...  # b) a number, 4 decimals
answer_B8_3c = ...  # c) a number, 4 decimals
answer_B8_3d = ...  # d) a number, 4 decimals
answer_B8_3e = ...  # e) True or False
answer_B8_3f = ...  # f) a number, 3 decimals
answer_B8_3g = ...  # g) a number, 4 decimals
answer_B8_3h = ...  # h) a number, 3 decimals

for letter, answer in zip("abcdefgh", [answer_B8_3a, answer_B8_3b, answer_B8_3c, answer_B8_3d, answer_B8_3e, answer_B8_3f, answer_B8_3g, answer_B8_3h]):
    wb.check(f"B8.3{letter}", answer)

**Ex B8.4 — GAE à la main sur quatre pas, pour λ = 0, 0,95 et 1**

In [ ]:
# Ex B8.4: the values you found on paper
answer_B8_4a = ...  # a) a number, 2 decimals
answer_B8_4b = ...  # b) a number, 2 decimals
answer_B8_4c = ...  # c) a number, 2 decimals
answer_B8_4d = ...  # d) a number, 4 decimals
answer_B8_4e = ...  # e) a number, 4 decimals
answer_B8_4f = ...  # f) a number, 4 decimals
answer_B8_4g = ...  # g) a number, 3 decimals
answer_B8_4h = ...  # h) a number, 3 decimals
answer_B8_4i = ...  # i) a number, 3 decimals
answer_B8_4j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_B8_4a, answer_B8_4b, answer_B8_4c, answer_B8_4d, answer_B8_4e, answer_B8_4f, answer_B8_4g, answer_B8_4h, answer_B8_4i, answer_B8_4j]):
    wb.check(f"B8.4{letter}", answer)

**Ex B8.5 — Objectif clippé de PPO : quatre cas, valeur et gradient nul ou non**

In [ ]:
# Ex B8.5: the values you found on paper
answer_B8_5a = ...  # a) a number, 2 decimals
answer_B8_5b = ...  # b) a number, 2 decimals
answer_B8_5c = ...  # c) a number, 2 decimals
answer_B8_5d = ...  # d) a number, 3 decimals
answer_B8_5e = ...  # e) a number, 3 decimals
answer_B8_5f = ...  # f) a number, 2 decimals
answer_B8_5g = ...  # g) a whole number
answer_B8_5h = ...  # h) a number, 2 decimals
answer_B8_5i = ...  # i) a whole number

for letter, answer in zip("abcdefghi", [answer_B8_5a, answer_B8_5b, answer_B8_5c, answer_B8_5d, answer_B8_5e, answer_B8_5f, answer_B8_5g, answer_B8_5h, answer_B8_5i]):
    wb.check(f"B8.5{letter}", answer)

**Ex B8.6 — Préférences : loss de Bradley-Terry puis loss DPO sur une paire de réponses**

In [ ]:
# Ex B8.6: the values you found on paper
answer_B8_6a = ...  # a) a number, 3 decimals
answer_B8_6b = ...  # b) a number, 3 decimals
answer_B8_6c = ...  # c) a number, 3 decimals
answer_B8_6d = ...  # d) a number, 3 decimals
answer_B8_6e = ...  # e) True or False
answer_B8_6f = ...  # f) a number, 2 decimals
answer_B8_6g = ...  # g) a number, 2 decimals
answer_B8_6h = ...  # h) a number, 3 decimals
answer_B8_6i = ...  # i) a number, 3 decimals
answer_B8_6j = ...  # j) a number, 4 decimals

for letter, answer in zip("abcdefghij", [answer_B8_6a, answer_B8_6b, answer_B8_6c, answer_B8_6d, answer_B8_6e, answer_B8_6f, answer_B8_6g, answer_B8_6h, answer_B8_6i, answer_B8_6j]):
    wb.check(f"B8.6{letter}", answer)

## Partie A · La valeur : un réseau à la place de la table

Au ch. 26, tu as tenu CartPole avec une table de 12 cases (26.38) et écrit `rl.py` : `ReplayBuffer`, `returns_to_go`, `evaluate_policy`. Ici, un **réseau** remplace la table. Tu prends d'abord CartPole en main pour un réseau (B8.12), tu écris ton DQN avec sa mémoire de rejeu et son réseau cible (B8.13), tu répares le DQN d'un collègue qui n'apprend rien (B8.14), puis tu mesures ce que chaque stabilisateur apporte, sur trois graines (B8.15).

La cellule suivante charge les outils du chapitre : `wbr` (`wb.deep_rl_env`), qui fabrique les environnements, évalue une politique (`wbr.evaluate`, 500 pas au plus, sans exploration) et contient les agents de **référence** dont se servent les vérifications ; `EVAL_SEED`, les graines de toutes les évaluations (jamais celles d'un entraînement) ; `TEST_DEEP_RL`, le fichier des tests de `deep_rl.py`. Ta librairie `rl.py` du ch. 26 sert telle quelle : si tu ne l'as pas, la référence la remplace (une ligne ℹ️ le dit). PyTorch tourne sur un seul thread : pour d'aussi petits réseaux, c'est plus rapide.

In [ ]:
# Tools of part A: CartPole, the evaluation and the reference agents of the chapter (wb.deep_rl_env, `wbr`)
import copy
import inspect
import math
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from wb import deep_rl_env as wbr
from wb.nbtools import StopCheck, checked, filled, fr, pair, refuses, returned, run_pytest_group, same, verdict, written

torch.set_num_threads(1)           # tiny networks: one thread is faster, and a seed gives the same numbers every time
SOURCE = "learner"                  # who trained the agents this notebook saves ("solution" in the solutions notebook)
TEST_DEEP_RL = "tests/test_chb8_deep_rl.py"   # the tests of your library deep_rl.py
EVAL_SEED = wbr.EVAL_SEED          # the seeds of every evaluation of the notebook (never used to train)
EPISODE_STEPS = wbr.EPISODE_STEPS  # 500: the time limit of CartPole-v1
RANDOM_LEVEL = 22.3                # the mean return of a random policy (fiche, 208.1)

_RL_PROBES = {
    "evaluate_policy": (lambda: mylearn.rl.evaluate_policy(wbr.make_env(), lambda s, i: 0, 1, max_steps=3,
                                                           rng=np.random.default_rng(0)), "26.21"),
    "ReplayBuffer": (lambda: (lambda b: (b.push(np.zeros(4, np.float32), 0, 1.0, np.zeros(4, np.float32), False),
                                         len(b), b.sample(1)))(mylearn.rl.ReplayBuffer(2, obs_shape=(4,),
                                                                                       obs_dtype="float32")), "26.31"),
    "returns_to_go": (lambda: mylearn.rl.returns_to_go([1.0, 1.0], 0.5), "26.14"),
}


def rl_ready(ex_id, *names):
    """True when the functions `names` of your mylearn/rl.py (chapter 26) are written; otherwise one ⏳ line that
    names them and their exercise of chapter 26."""
    missing = [name for name in names if not written(_RL_PROBES[name][0])]
    if missing:
        where = ", ".join(f"{name} (Ex {_RL_PROBES[name][1]})" for name in missing)
        print(f"⏳ Ex {ex_id} : cet exercice se sert de ta librairie rl.py du ch. 26 : {where}, pas encore écrit.")
    return not missing


def plot_returns(ax, returns, label, ends=None, window=20, **kwargs):
    """The returns of the training episodes as a moving average of `window` episodes (against the episode number, or
    against the environment step of their end when `ends` is given)."""
    returns = np.asarray(returns, dtype=float)
    x = np.arange(1, len(returns) + 1) if ends is None else np.asarray(ends)
    ax.plot(x, wbr.moving_average(returns, window), label=label, **kwargs)


print(f"CartPole-v1 (gymnasium {gym.__version__}) : {EPISODE_STEPS} pas au plus, résolu à {wbr.SOLVED:g} de moyenne sur "
      f"100 épisodes · évaluations sur les graines tirées de {EVAL_SEED} · PyTorch {torch.__version__} sur un thread")

### Ex B8.12 — CartPole pour le deep RL : espaces, politique aléatoire, evaluate_policy et la référence tabulaire du ch. 26 📦 ★ ⏱️ 15 min
**Objectif :** lire l'interface de CartPole et évaluer honnêtement une politique, aléatoire ou tabulaire, avant d'y mettre un réseau.  
**Prérequis :** ch. 26 (26.21, 26.36, 26.38) · fiche §208.1 (« CartPole, vu de près »), §208.6 (« Évaluer un agent », piège « un plafond caché dans l'évaluation ») · **Fil rouge :** CartPole · **Parcours :** R, C

**Contrat** des trois fonctions de la cellule suivante :
- `describe_12(env)` rend un dictionnaire **lu dans `env`**, jamais tapé : `"obs_shape"`, la forme d'une observation (un tuple d'entiers) ; `"n_actions"`, le nombre d'actions (un `int`) ; `"max_steps"`, la limite de temps d'un épisode (un `int`, dans `env.spec`) ; `"threshold"`, le retour moyen qui compte comme résolu (un `float`, dans `env.spec`) ;
- `random_policy_12(seed)` rend une politique `(state, info) -> int` qui joue 0 ou 1 au hasard : **chaque appel** tire `int(rng.integers(2))` d'**un seul** générateur `rng = np.random.default_rng(seed)`, créé une fois par `random_policy_12` (pas à chaque appel). `env.action_space.sample()` ne convient pas : `reset(seed=...)` ne réinitialise pas son générateur (🧠 Q10) ;
- `evaluate_12(env, policy, seed, n_episodes=100)` rend le dictionnaire de `mylearn.rl.evaluate_policy` (ta fonction de 26.21) pour `n_episodes` épisodes **entiers** de `env` : `max_steps` vaut la limite de temps de `env` (dans `env.spec`), pas le défaut d'`evaluate_policy` ; les graines des épisodes viennent de `rng=np.random.default_rng(seed)` (un générateur, jamais un entier).

La cellule donnée fixe `SEED_12` et relit la **table du ch. 26** (la solution de 26.38 : 12 états, l'angle et sa vitesse) : `MAKE_TABULAR_12()` fabrique le CartPole discrétisé, `TABULAR_POLICY_12` est sa politique gloutonne (une action par état), sans réentraînement.

La vérification essaie `describe_12` sur CartPole-v1 et sur un CartPole coupé à 200 pas, `random_policy_12` sur 40 appels, `evaluate_12` sur la règle d'une ligne de 208.1, puis évalue ta politique aléatoire (graine `SEED_12`, 100 épisodes) et la table du ch. 26, avec et sans `max_steps`. Le retour moyen de ta politique aléatoire est vérifié par `wb.check` : rien à taper, aucune valeur à arrondir.

Dans tes notes : pourquoi la moyenne de ta politique aléatoire n'est-elle pas celle de la fiche (22,3) ? De combien varie une moyenne de 100 épisodes d'une graine à l'autre (ch. 2 : l'erreur type) ? Que deviendrait le score de la table si l'on oubliait `max_steps` ? Combien de cases faudrait-il à une table qui découperait les **quatre** variables en dix intervalles, et que perd-on à n'en garder que deux ?

In [ ]:
SEED_12 = 13                                    # the seed of the random policy and of the episodes
MAKE_TABULAR_12, Q_TABULAR_12 = wbr.tabular_reference()   # the table of Ex 26.38 (12 states), read back, not trained
TABULAR_POLICY_12 = Q_TABULAR_12.argmax(axis=1)            # its greedy policy: one action per state (a table)

In [ ]:
def describe_12(env):
    """What the agent sees and may do in env, READ from env (never typed): a dict {"obs_shape": the shape of one
    observation (a tuple of ints), "n_actions": the number of actions (an int), "max_steps": the time limit of an
    episode (an int, from env.spec), "threshold": the mean return that counts as solved (a float, from env.spec)}."""
    raise NotImplementedError("describe_12")


def random_policy_12(seed):
    """A policy (state, info) -> int that plays 0 or 1 at random: each call draws int(rng.integers(2)) from ONE
    generator rng = np.random.default_rng(seed), created once by random_policy_12 (not at each call)."""
    raise NotImplementedError("random_policy_12")


def evaluate_12(env, policy, seed, n_episodes=100):
    """The dict of mylearn.rl.evaluate_policy for n_episodes WHOLE episodes of env (max_steps = the time limit of env,
    read in env.spec, not the default of evaluate_policy), the seeds of the episodes drawn from
    rng=np.random.default_rng(seed)."""
    raise NotImplementedError("evaluate_12")

In [ ]:
with checked("B8.12"):
    info_12 = describe_12(gym.make("CartPole-v1"))
    if returned("B8.12", "describe_12", info_12):
        want_12 = {"obs_shape": (4,), "n_actions": 2, "max_steps": 500, "threshold": 475.0}
        if not isinstance(info_12, dict) or set(info_12) != set(want_12):
            print(f"❌ Ex B8.12 : describe_12 doit rendre un dict aux clés {sorted(want_12)}, reçu "
                  f"{sorted(info_12) if isinstance(info_12, dict) else type(info_12).__name__}.")
        else:
            got_12 = {k: (tuple(v) if k == "obs_shape" and not isinstance(v, str) else v) for k, v in info_12.items()}
            verdict("B8.12", got_12 == want_12 and isinstance(got_12["n_actions"], int),
                    "describe_12 lit les espaces et env.spec de CartPole-v1.",
                    f"describe_12 sur CartPole-v1 : reçu {got_12} ; relis 208.1 (« CartPole, vu de près »).")
            short_12 = describe_12(gym.make("CartPole-v1", max_episode_steps=200))
            verdict("B8.12", short_12.get("max_steps") == 200,
                    "la limite de temps vient bien de l'environnement (200 pour gym.make(..., max_episode_steps=200)).",
                    f"pour gym.make(\"CartPole-v1\", max_episode_steps=200), describe_12 rend max_steps = "
                    f"{short_12.get('max_steps')!r} : lit-elle vraiment env.spec ?")

with checked("B8.12"):
    actions_12 = [random_policy_12(SEED_12)(None, {}) for _ in range(3)]
    policy_12 = random_policy_12(SEED_12)
    stream_12 = [policy_12(None, {}) for _ in range(40)]
    want_12 = [int(a) for a in np.random.default_rng(SEED_12).integers(2, size=40)]
    if not all(type(a) is int for a in stream_12):
        print(f"❌ Ex B8.12 : la politique doit rendre un int Python (gymnasium refuse un tenseur), reçu "
              f"{type(stream_12[0]).__name__}.")
    elif stream_12 != want_12:
        constant_12 = len(set(stream_12)) == 1
        print("❌ Ex B8.12 : " + ("40 appels de la politique rendent toujours la même action : le générateur "
                                  "repart-il de sa graine à chaque appel ?" if constant_12 else
                                  "les 40 premières actions ne sont pas les tirages int(rng.integers(2)) de "
                                  "np.random.default_rng(seed) : relis le contrat."))
    else:
        verdict("B8.12", True, "random_policy_12 tire ses actions d'un seul générateur à graine.", "")

with checked("B8.12"):
    if rl_ready("B8.12", "evaluate_policy"):
        rule_12 = evaluate_12(gym.make("CartPole-v1"), wbr.rule_policy, 1, n_episodes=5)
        if returned("B8.12", "evaluate_12", rule_12):
            verdict("B8.12", rule_12["mean_length"] == EPISODE_STEPS,
                    "la règle d'une ligne de 208.1 tient 500 pas : tes épisodes vont jusqu'à la limite de CartPole-v1.",
                    f"la règle d'une ligne de 208.1 tient {rule_12['mean_length']:g} pas en moyenne au lieu de 500 : "
                    "tes épisodes s'arrêtent-ils avant la limite de temps de l'environnement ? (piège de 208.6)")
            random_12 = evaluate_12(gym.make("CartPole-v1"), random_policy_12(SEED_12), SEED_12)
            tab_12 = evaluate_12(MAKE_TABULAR_12(), TABULAR_POLICY_12, 26)
            tab_short_12 = mylearn.rl.evaluate_policy(MAKE_TABULAR_12(), TABULAR_POLICY_12, 100,
                                                      rng=np.random.default_rng(26))
            print(f"politique aléatoire (graine {SEED_12}) : {fr(random_12['mean_return'], 2)} de retour moyen sur 100 "
                  f"épisodes (écart-type {fr(random_12['std_return'], 1)})")
            print(f"table du ch. 26 (12 états, 26.38) : {fr(tab_12['mean_return'], 1)} sur 100 épisodes entiers ; "
                  f"{fr(tab_short_12['mean_return'], 1)} avec le max_steps par défaut d'evaluate_policy")
            wb.check("B8.12", random_12["mean_return"], computed=True)

### Ex B8.13 — DQN sur CartPole : réseau Q, ReplayBuffer de rl.py, réseau cible, Huber, ε décroissant 📦 ★★★ ⏱️ 60 min
**Objectif :** écrire un DQN complet en PyTorch, avec la mémoire de rejeu de ta librairie, un réseau cible, la loss de Huber et un ε décroissant, et vérifier qu'il apprend.  
**Prérequis :** Ex B8.12 · ✏️ B8.1, B8.2 · 🔨 26.31 (`ReplayBuffer`) · fiche §208.1 (« Apprendre une valeur par régression »), §208.2 · **Fil rouge :** CartPole · **Parcours :** R, C

**Contrat** des quatre fonctions de la cellule suivante (les réglages sont dans `SETTINGS_13`, ceux du DQN de référence du chapitre ; la cellule donnée les affiche) :
- `make_q_net_13(hidden=64)` rend un `nn.Module` neuf : 4 entrées, deux couches cachées de `hidden` neurones avec ReLU, 2 sorties (une valeur par action), en float32 ;
- `dqn_targets_13(q_target, next_states, rewards, terminated, gamma)` rend les cibles d'un mini-batch, un tenseur de forme `(B,)` **sans gradient** : $r + \gamma(1 - d)\max_{a'} Q_{\text{cible}}(s', a')$, avec `next_states` float32 `(B, 4)`, `rewards` `(B,)` et `terminated` booléen `(B,)` ;
- `epsilon_13(step, total_steps, settings=SETTINGS_13)` rend l'ε du pas `step` (0, 1, …) : linéaire de `eps_start` à `eps_end` sur les `eps_fraction × total_steps` premiers pas, puis `eps_end` ;
- `train_dqn_13(env, seed, total_steps, settings=SETTINGS_13)` entraîne un DQN pendant exactement `total_steps` pas de `env` et rend le triplet `(q_net, buffer, history)` :
  - **graines** : `torch.manual_seed(seed)` puis le réseau ; `env.reset(seed=seed)` au premier épisode seulement ; un générateur `np.random.default_rng(seed)` pour l'exploration ; `random_state=seed` pour la mémoire. Deux appels de même graine rendent le même réseau et le même historique ;
  - **la mémoire** : `mylearn.rl.ReplayBuffer(settings["capacity"], obs_shape=(4,), obs_dtype="float32", random_state=seed)`, ta classe du ch. 26 ; on y range `(s, a, r, s', terminated)`, **`terminated` seul** ;
  - **chaque pas** : action ε-greedy (`epsilon_13`), un pas d'environnement, la transition rangée ; un épisode **terminé ou tronqué** se termine par un `env.reset()` ;
  - **les mises à jour** : dès que `step + 1 >= learning_starts` et que la mémoire contient un mini-batch, tous les `train_freq` pas : un mini-batch de `batch_size`, les cibles de `dqn_targets_13` avec le réseau cible, la loss `F.huber_loss(Q(s, a), y, delta=huber_delta)`, un pas d'Adam (`lr`) après `nn.utils.clip_grad_norm_(..., max_grad_norm)` ;
  - **le réseau cible** : une copie du réseau en ligne au départ, réécrite (`load_state_dict`) tous les `target_every` pas ;
  - **`history`** : `{"returns": [...], "ends": [...]}`, le retour de chaque épisode fini et le nombre de pas joués quand il a fini (1, 2, …).

La vérification essaie `make_q_net_13`, les cibles sur 4 transitions fixées (dont une terminée), ε à cinq pas, puis un entraînement de 600 pas sur un CartPole **coupé à 15 pas** (beaucoup de limites de temps) : le nombre de pas joués, ta mémoire (sa capacité, ce qu'elle range comme fins), l'historique et la reproductibilité. La cellule suivante entraîne ton DQN sur `STEPS_13` pas (20 000 en mode rapide, ≈ 40 s ; 50 000 en mode complet), trace les retours d'entraînement et évalue la politique **gloutonne** à part, sur 100 épisodes (`wbr.evaluate`, 500 pas au plus). Elle attend deux propriétés, pas un nombre exact : le retour d'entraînement monte, et la politique gloutonne dépasse `THRESHOLD_13`.

Dans tes notes : combien d'épisodes ton DQN a-t-il joués, et combien de mises à jour a-t-il faites ? Pourquoi l'évaluation gloutonne diffère-t-elle des retours d'entraînement de la fin ? Pourquoi la courbe monte-t-elle puis redescend-elle parfois ? Que changerait `F.smooth_l1_loss` à la place de `F.huber_loss` avec ce `huber_delta` (✏️ B8.2) ?

In [ ]:
SETTINGS_13 = dict(wbr.DQN_SETTINGS)   # the settings of the reference DQN of the chapter (printed below)
STEPS_13 = wbr.dqn_steps(FAST_MODE)       # 20 000 environment steps in fast mode, 50 000 in full mode
SEED_13 = 13
THRESHOLD_13 = 55             # the greedy evaluation that the check asks (calibrated on 8 seeds of the reference)
print(SETTINGS_13)

In [ ]:
def make_q_net_13(hidden=64):
    """A new Q network for CartPole: 4 inputs (the observation), two hidden layers of `hidden` neurons with ReLU, 2
    outputs (one value per action), float32."""
    raise NotImplementedError("make_q_net_13")


def dqn_targets_13(q_target, next_states, rewards, terminated, gamma):
    """The DQN targets of a mini-batch, a tensor (B,) that carries NO gradient:
    r + gamma (1 - terminated) max_a' Q_target(s', a'). next_states: float32 (B, 4); rewards: float (B,); terminated:
    bool (B,)."""
    raise NotImplementedError("dqn_targets_13")


def epsilon_13(step, total_steps, settings=SETTINGS_13):
    """ε of the step `step` (0, 1, ...): linear from settings["eps_start"] to settings["eps_end"] over the first
    settings["eps_fraction"] * total_steps steps, then settings["eps_end"]."""
    raise NotImplementedError("epsilon_13")


def train_dqn_13(env, seed, total_steps, settings=SETTINGS_13):
    """Train a DQN for total_steps steps of env (the contract of the statement); return (q_net, buffer, history), with
    history = {"returns": the return of each finished episode, "ends": the number of steps done (1, 2, ...) when each
    of them ended}."""
    raise NotImplementedError("train_dqn_13")

In [ ]:
ok_13 = False
with checked("B8.13"):
    net_13 = make_q_net_13()
    if returned("B8.13", "make_q_net_13", net_13):
        out_13 = net_13(torch.zeros(5, 4)) if isinstance(net_13, nn.Module) else None
        verdict("B8.13", out_13 is not None and tuple(out_13.shape) == (5, 2) and out_13.dtype == torch.float32,
                f"make_q_net_13 : {sum(p.numel() for p in net_13.parameters())} paramètres, une sortie (5, 2) pour 5 "
                "observations.",
                f"make_q_net_13 : attendu un nn.Module qui rend (5, 2) en float32 pour 5 observations, reçu "
                f"{type(net_13).__name__}" + (f" de sortie {tuple(out_13.shape)} {out_13.dtype}" if out_13 is not None
                                              else "") + ".")

targets_ok_13 = False
with checked("B8.13"):
    probe_13 = nn.Linear(4, 2)                 # Q_target(s', .) = the first two numbers of s'
    with torch.no_grad():
        probe_13.weight.copy_(torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]]))
        probe_13.bias.zero_()
    next_13 = torch.tensor([[2.0, 3.0, 0.3, 0.0], [5.0, -1.0, 0.0, 0.2], [0.5, 0.25, 0.1, 0.1], [-1.0, -2.0, 0.0, 0.0]])
    rewards_13 = torch.tensor([1.0, 0.0, 1.0, -1.0])
    term_13 = torch.tensor([False, True, False, False])
    y_13 = dqn_targets_13(probe_13, next_13.clone(), rewards_13.clone(), term_13.clone(), 0.9)
    if returned("B8.13", "dqn_targets_13", y_13):
        if getattr(y_13, "requires_grad", False):
            print("❌ Ex B8.13 : les cibles portent un gradient (requires_grad=True) : la loss serait dérivée aussi à "
                  "travers elles ; relis « Apprendre une valeur par régression » (208.1).")
        else:
            targets_ok_13 = same("B8.13", "dqn_targets_13 sur 4 transitions (la 2ᵉ terminée, γ = 0,9)", y_13,
                                 rewards_13 + 0.9 * (~term_13).float() * torch.tensor([3.0, 5.0, 0.5, -1.0]),
                                 rtol=1e-6, atol=1e-6)

eps_ok_13 = False
with checked("B8.13"):
    eps_steps_13 = [0, 1500, 3000, 6000, 9999]
    eps_ok_13 = same("B8.13", "epsilon_13 aux pas 0, 1 500, 3 000, 6 000 et 9 999 d'un entraînement de 10 000 pas",
         [epsilon_13(k, 10_000) for k in eps_steps_13],
         [wbr.linear_epsilon(k, 10_000, 1.0, 0.05, SETTINGS_13["eps_fraction"]) for k in eps_steps_13])

with checked("B8.13"):
    if not targets_ok_13:
        print("⏳ Ex B8.13 : l'essai de train_dqn_13 attend des cibles justes (dqn_targets_13, ci-dessus).")
    elif rl_ready("B8.13", "ReplayBuffer"):
        small_13 = {**SETTINGS_13, "learning_starts": 200, "capacity": 1_000, "batch_size": 32, "target_every": 100}
        counter_13 = wbr.StepCounter(wbr.make_env(max_episode_steps=15))   # many episodes cut by a 15-step time limit
        q_a_13, buf_a_13, hist_a_13 = pair("B8.13", "train_dqn_13", train_dqn_13(counter_13, 0, 600, small_13),
                                           names=("q_net", "buffer", "history"))
        good_13 = []
        good_13.append(counter_13.steps == 600)
        verdict("B8.13", good_13[-1], "600 pas d'environnement pour total_steps = 600.",
                f"train_dqn_13(env, 0, 600, ...) a joué {counter_13.steps} pas d'environnement au lieu de 600.")
        is_buffer_13 = isinstance(buf_a_13, mylearn.rl.ReplayBuffer)
        good_13.append(is_buffer_13 and buf_a_13.capacity == 1_000 and len(buf_a_13) == 600)
        verdict("B8.13", good_13[-1], "la mémoire est une ReplayBuffer de ta librairie : capacité 1 000, 600 transitions.",
                "attendu en deuxième sortie la ReplayBuffer de mylearn.rl, de capacité settings[\"capacity\"] (1 000) "
                "et qui contient les 600 transitions, reçu " + (f"une ReplayBuffer de capacité {buf_a_13.capacity} et "
                f"{len(buf_a_13)} transitions" if is_buffer_13 else type(buf_a_13).__name__) + ".")
        if is_buffer_13:
            stored_13 = int(np.sum(buf_a_13._terminated[:min(len(buf_a_13), buf_a_13.capacity)]))
            good_13.append(stored_13 == counter_13.terminations)
            verdict("B8.13", good_13[-1],
                    f"la mémoire range {stored_13} fins d'épisode : les {counter_13.terminations} vraies fins, pas les "
                    f"{counter_13.truncations} limites de temps.",
                    f"la mémoire range {stored_13} transitions marquées comme fin, pour {counter_13.terminations} vraies "
                    f"fins (terminated) et {counter_13.truncations} limites de temps (truncated) ; relis l'encadré ⚠️ "
                    "« la limite de temps n'est pas une fin » (208.2).")
        returns_a_13 = list(hist_a_13.get("returns", [])) if isinstance(hist_a_13, dict) else None
        ends_a_13 = list(hist_a_13.get("ends", [])) if isinstance(hist_a_13, dict) else None
        n_ends_13 = counter_13.terminations + counter_13.truncations
        good_13.append(returns_a_13 is not None and len(returns_a_13) == len(ends_a_13) == n_ends_13
                       and ends_a_13 == sorted(ends_a_13) and (not ends_a_13 or (ends_a_13[-1] <= 600
                       and abs(sum(returns_a_13) - ends_a_13[-1]) < 1e-9)))
        verdict("B8.13", good_13[-1], f"history : {n_ends_13} épisodes finis, leurs retours et leurs fins.",
                f"history doit avoir \"returns\" et \"ends\", une valeur par épisode fini ({n_ends_13} ici), les fins "
                "comptées en pas d'environnement (1, 2, …) et la somme des retours égale au pas de la dernière fin ; "
                f"reçu {len(returns_a_13) if returns_a_13 is not None else '?'} retours.")
        counter_b_13 = wbr.StepCounter(wbr.make_env(max_episode_steps=15))
        q_b_13, _, hist_b_13 = train_dqn_13(counter_b_13, 0, 600, small_13)
        probe_states_13 = torch.as_tensor(np.random.default_rng(13).normal(0, 0.1, (64, 4)), dtype=torch.float32)
        with torch.no_grad():
            same_q_13 = torch.equal(q_a_13(probe_states_13), q_b_13(probe_states_13))
        good_13.append(same_q_13 and hist_b_13 == hist_a_13)
        verdict("B8.13", good_13[-1], "deux entraînements de même graine donnent le même réseau et le même historique.",
                "deux entraînements de même graine ne donnent pas le même " + ("historique" if same_q_13 else "réseau")
                + " : relis le contrat des graines (torch.manual_seed, le premier reset, le générateur de "
                "l'exploration, random_state de la mémoire).")
        ok_13 = all(good_13) and eps_ok_13

In [ ]:
if not ok_13:
    print("⏳ Ex B8.13 : l'entraînement complet attend que la vérification ci-dessus passe.")
else:
    with checked("B8.13"):
        start_13 = time.perf_counter()
        Q_NET_13, BUFFER_13, HISTORY_13 = train_dqn_13(wbr.make_env(), SEED_13, STEPS_13)
        seconds_13 = time.perf_counter() - start_13
        returns_13 = np.asarray(HISTORY_13["returns"], dtype=float)
        eval_13 = wbr.evaluate(wbr.greedy_q(Q_NET_13), 100, EVAL_SEED)
        fig_13, ax_13 = plt.subplots(figsize=(7, 3))
        ax_13.plot(HISTORY_13["ends"], returns_13, ".", alpha=0.25, label="un épisode d'entraînement")
        plot_returns(ax_13, returns_13, "moyenne glissante sur 20 épisodes", ends=HISTORY_13["ends"])
        ax_13.axhline(eval_13["mean_return"], color="k", ls="--", lw=1, label="évaluation gloutonne finale")
        ax_13.set(xlabel="pas d'environnement", ylabel="retour", title=f"Ton DQN, graine {SEED_13}")
        ax_13.legend(fontsize=8)
        plt.show()
        print(f"{STEPS_13} pas en {seconds_13:.0f} s, {len(returns_13)} épisodes · 20 premiers épisodes : "
              f"{fr(returns_13[:20].mean(), 1)} · 20 derniers : {fr(returns_13[-20:].mean(), 1)} · politique gloutonne "
              f"sur 100 épisodes : {fr(eval_13['mean_return'], 1)} (écart-type {fr(eval_13['std_return'], 1)})")
        verdict("B8.13", returns_13[-20:].mean() >= 2 * returns_13[:20].mean(),
                "le retour d'entraînement monte : les 20 derniers épisodes valent au moins le double des 20 premiers.",
                "le retour d'entraînement ne monte pas : les 20 derniers épisodes ne valent pas le double des 20 premiers.")
        verdict("B8.13", eval_13["mean_return"] >= THRESHOLD_13,
                f"la politique gloutonne dépasse {THRESHOLD_13:g} (la référence, sur 8 graines, fait au moins "
                f"93).",
                f"la politique gloutonne tient {fr(eval_13['mean_return'], 1)} pas en moyenne : moins que les "
                f"{THRESHOLD_13:g} attendus (la référence, sur 8 graines, fait au moins 93).")

### Ex B8.14 — Le DQN qui diverge : quatre erreurs de cible, de fin d'épisode et d'exploration 🐛 ★★ ⏱️ 30 min
**Objectif :** retrouver par des diagnostics enchaînés les quatre erreurs d'un DQN qui n'apprend rien, sans relancer d'entraînement à chaque essai.  
**Prérequis :** Ex B8.13 · 🐛 26.24 · 🧠 Q3 · fiche §208.1, §208.2 (pièges) · **Fil rouge :** CartPole · **Parcours :** C

Un collègue a écrit un DQN en quatre morceaux : `new_memory(seed)`, la mémoire de rejeu (ta `ReplayBuffer`) ; `remember(...)`, qui range une transition et dit si l'épisode est fini ; `exploration_rate(step)`, le planning d'ε ; `bellman_targets(online, frozen, batch)`, les cibles d'un mini-batch ; et quelques constantes. Sa boucle d'entraînement, `run_colleague_14` (cellule donnée), est juste. Son DQN n'apprend rien : quatre erreurs se sont glissées dans les morceaux, celles du titre. **Répare la cellule suivante en place** (mêmes noms, mêmes signatures).

La cellule de vérification enchaîne quatre diagnostics, chacun sur un morceau, sans entraîner : 1. les cibles d'un mini-batch fixé, avec deux réseaux différents ; 2. ce que `remember` range pour une limite de temps, une vraie fin et un pas ordinaire ; 3. ε à quatre moments du planning ; 4. de quand datent les transitions d'un mini-batch, après 2 000 pas rangés. Le premier qui échoue arrête la chaîne (⏳ pour les suivants) ; un message dit ce qu'il mesure, pas ce qu'il faut changer. La cellule d'expérience entraîne le DQN tel qu'il est (`STEPS_14` pas, une quinzaine de secondes) et trace ses retours, son ε et la moyenne de $\max_a Q$ sur 256 états de départ : lance-la une première fois **avant** de réparer, pour voir les symptômes. Une fois les quatre diagnostics passés, elle vérifie que le DQN réparé apprend (politique gloutonne au-dessus de `THRESHOLD_14`).

Dans tes notes : pour chaque erreur, le symptôme que tu as vu (sur les courbes, puis dans le message) et la ligne fautive. Lesquelles se voyaient sur les courbes, et lesquelles seulement dans un diagnostic ? Pourquoi un diagnostic par morceau plutôt qu'un entraînement complet après chaque correction ? Laquelle de ces erreurs garderait un DQN qui « apprend un peu » ?

In [ ]:
STEPS_14 = 8_000 if FAST_MODE else 20_000
THRESHOLD_14 = 90       # the greedy evaluation of the repaired DQN (calibrated on 5 seeds of the repair)


def run_colleague_14(seed, total_steps):
    """The training loop of the colleague (right as it is): it calls new_memory, remember, exploration_rate and
    bellman_targets and the constants of the cell below. Returns (online, log): log["returns"] and log["ends"] (the
    finished episodes), log["eps"] (ε every 100 steps) and log["q_mean"] (the mean of max Q over 256 fixed states,
    every 500 steps)."""
    torch.manual_seed(seed)
    online = wbr.q_network(64)
    frozen = copy.deepcopy(online)
    optimizer = torch.optim.Adam(online.parameters(), lr=LEARNING_RATE)
    rng = np.random.default_rng(seed)
    memory = new_memory(seed)
    env = wbr.make_env()
    obs, _ = env.reset(seed=seed)
    probe = torch.as_tensor(np.random.default_rng(14).uniform(-0.05, 0.05, (256, 4)), dtype=torch.float32)
    log = {"returns": [], "ends": [], "eps": [], "q_mean": []}
    episode_return = 0.0
    for step in range(total_steps):
        eps = exploration_rate(step)
        if step % 100 == 0:
            log["eps"].append(float(eps))
        if rng.random() < eps:
            action = int(rng.integers(2))
        else:
            with torch.no_grad():
                action = int(online(torch.as_tensor(obs).unsqueeze(0)).argmax(dim=1))
        next_obs, reward, term, trunc, _ = env.step(action)
        over = remember(memory, obs, action, reward, next_obs, term, trunc)
        episode_return += reward
        obs = next_obs
        if over:
            log["returns"].append(episode_return)
            log["ends"].append(step + 1)
            episode_return = 0.0
            obs, _ = env.reset()
        if step >= 500 and len(memory) >= MINI_BATCH:
            batch = memory.sample(MINI_BATCH)
            y = bellman_targets(online, frozen, batch)
            q_sa = online(torch.as_tensor(batch["states"])).gather(1, torch.as_tensor(batch["actions"])[:, None])
            loss = F.huber_loss(q_sa.squeeze(1), y)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online.parameters(), 10.0)
            optimizer.step()
        if (step + 1) % 500 == 0:
            frozen.load_state_dict(online.state_dict())
            with torch.no_grad():
                log["q_mean"].append(float(online(probe).max(dim=1).values.mean()))
    return online, log


class _SpyMemory_14:
    """Records what remember stores (point 2), without a real memory."""

    def __init__(self):
        self.pushed = []

    def push(self, state, action, reward, next_state, terminated):
        self.pushed.append(terminated)


def diagnose_14():
    """Four diagnoses of the colleague's pieces, in a row: the first that fails stops the chain. True when they pass."""
    rng = np.random.default_rng(140)
    states = rng.uniform(-0.1, 0.1, (5, 4)).astype(np.float32)
    next_states = rng.uniform(-0.1, 0.1, (5, 4)).astype(np.float32)
    batch = {"states": states, "actions": np.array([0, 1, 1, 0, 1]), "rewards": np.ones(5),
             "next_states": next_states, "terminated": np.array([False, False, True, False, False])}

    def point1():
        online, frozen = wbr.q_network(16, seed=1), wbr.q_network(16, seed=2)
        y = bellman_targets(online, frozen, {k: v.copy() for k, v in batch.items()})
        if getattr(y, "requires_grad", False):
            return False, ("les cibles de 5 transitions portent un gradient (requires_grad=True) : loss.backward() "
                           "dériverait aussi à travers elles"), \
                "relis « Apprendre une valeur par régression » (208.1) : qu'est la cible pendant le calcul du gradient ?"
        want = wbr.dqn_targets(online, frozen, torch.as_tensor(next_states), torch.ones(5),
                               torch.as_tensor(batch["terminated"]), DISCOUNT)
        y = torch.as_tensor(y, dtype=torch.float32).reshape(-1)
        if y.shape == want.shape and torch.allclose(y, want, atol=1e-6):
            return True, "les cibles sont constantes pour le gradient et viennent du réseau figé", ""
        moved = wbr.q_network(16, seed=3)
        y_moved = torch.as_tensor(bellman_targets(moved, frozen, {k: v.copy() for k, v in batch.items()})).reshape(-1)
        if y_moved.shape == y.shape and not torch.allclose(y_moved, y):
            return False, (f"les cibles bougent (jusqu'à {fr(float((y_moved - y).abs().max()), 3)}) quand seuls les poids "
                           "du réseau en ligne changent"), \
                "relis « Les deux stabilisateurs » (208.2) : quel réseau calcule la cible, et pourquoi ?"
        return False, f"les cibles s'écartent de r + γ(1 − terminated) max Q(s′, ·)", "relis bellman_targets ligne à ligne"

    def point2():
        cases = [(False, True, "une limite de temps (terminated=False, truncated=True)"),
                 (True, False, "une vraie fin (terminated=True, truncated=False)"),
                 (False, False, "un pas ordinaire")]
        for term, trunc, what in cases:
            spy = _SpyMemory_14()
            over = remember(spy, states[0], 1, 1.0, next_states[0], term, trunc)
            if len(spy.pushed) != 1:
                return False, f"pour {what}, remember range {len(spy.pushed)} transitions au lieu d'une", \
                    "relis remember"
            if bool(over) != (term or trunc):
                return False, (f"pour {what}, remember rend {bool(over)} : la boucle " +
                               ("ne remettrait pas l'environnement à zéro" if term or trunc else
                                "remettrait l'environnement à zéro en plein épisode")), "relis sa docstring"
            if bool(spy.pushed[0]) != term:
                return False, (f"{what} entre dans la mémoire avec le drapeau de fin {bool(spy.pushed[0])}" +
                               (" : sa cible n'amorcera pas sur l'état suivant" if trunc else "")), \
                    "relis l'encadré ⚠️ « la limite de temps n'est pas une fin » (208.2)"
        return True, "seule une vraie fin entre dans la mémoire comme une fin ; les deux fins remettent à zéro", ""

    def point3():
        steps = [0, EPS_STEPS // 2, EPS_STEPS, 2 * EPS_STEPS]
        got = [float(exploration_rate(k)) for k in steps]
        want = [wbr.linear_epsilon(k, EPS_STEPS, EPS_FIRST, EPS_LAST, 1.0) for k in steps]
        if np.allclose(got, want, atol=1e-9):
            return True, f"ε passe de {fr(got[0], 2)} à {fr(got[2], 2)} en {EPS_STEPS} pas", ""
        shown = ", ".join(f"{fr(v, 3)} au pas {k}" for v, k in zip(got, steps))
        if max(got) - min(got) < 1e-9:
            return False, f"ε vaut {shown} : la politique ne quitte jamais le même niveau d'exploration", \
                "que doit valoir ε à la fin annoncée par la docstring ?"
        return False, f"ε vaut {shown}, ce qui ne suit pas la docstring", "relis exploration_rate ligne à ligne"

    def point4():
        memory = new_memory(0)
        for k in range(2000):
            memory.push(np.array([k, 0, 0, 0], np.float32), 0, 1.0, np.array([k + 1, 0, 0, 0], np.float32), False)
        drawn = np.concatenate([memory.sample(MINI_BATCH)["states"][:, 0] for _ in range(20)])
        recent = float(np.mean(drawn >= 1900))
        if recent <= 0.5:
            return True, (f"après 2 000 pas, {fr(100 * recent, 0)} % des transitions tirées datent des 100 derniers pas "
                          "(les mini-batches mêlent des moments éloignés)"), ""
        return False, (f"après 2 000 pas, {fr(100 * recent, 0)} % des transitions de 20 mini-batches de {MINI_BATCH} "
                       "datent des 100 derniers pas, soit deux épisodes d'un agent qui tient 50 pas : chaque mini-batch "
                       "reste fait de pas qui se suivent"), \
            "relis « Ces nombres montrent surtout… » (208.2) et 🧠 Q3 : que doit garder la mémoire ?"

    points = [("la cible", point1), ("les fins d'épisode", point2), ("l'exploration", point3), ("la mémoire", point4)]
    for k, (name, run) in enumerate(points, start=1):
        try:
            ok, measured, why = run()
        except NotImplementedError as error:
            print(f"⏳ Ex B8.14 : point {k}, {name} : pas encore fait ({error}) : la mémoire est ta ReplayBuffer "
                  "(rl.py, Ex 26.31).")
            return False
        except Exception as error:  # noqa: BLE001 - the colleague's code, as repaired: one ❌ line, the chain stops
            ok, measured, why = False, f"une erreur {type(error).__name__} : {str(error)[:200]}", "relis la fonction en cause"
        if ok:
            print(f"✅ Ex B8.14 : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex B8.14 : point {k}, {name} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex B8.14 : point {later} : corrige d'abord le point {k}.")
            return False
    print("✅ Ex B8.14 : les quatre diagnostics passent.")
    return True

In [ ]:
# The DQN of a colleague, to repair IN PLACE (same names, same signatures). It learns nothing on CartPole; four
# mistakes slipped into its pieces (run_colleague_14, the loop of the cell above that calls them, is right).
LEARNING_RATE = 5e-4
MINI_BATCH = 64
MEMORY_SLOTS = 100
DISCOUNT = 0.99
EPS_FIRST, EPS_LAST, EPS_STEPS = 1.0, 0.05, 3_000


def new_memory(seed):
    """The replay memory of the agent: your mylearn.rl.ReplayBuffer, for CartPole observations."""
    return mylearn.rl.ReplayBuffer(MEMORY_SLOTS, obs_shape=(4,), obs_dtype="float32", random_state=seed)


def remember(memory, obs, action, reward, next_obs, term, trunc):
    """Store one transition in the memory; return True when the episode is over (the loop then resets the env)."""
    finished = term | trunc
    memory.push(obs, action, reward, next_obs, finished)
    return finished


def exploration_rate(step):
    """ε of a step: from EPS_FIRST down to EPS_LAST, linearly over the first EPS_STEPS steps, then EPS_LAST."""
    weight = max(0.0, 1.0 - step / EPS_STEPS)          # 1 at the start, 0 after EPS_STEPS steps
    return weight * EPS_FIRST + (1.0 - weight) * EPS_FIRST


def bellman_targets(online, frozen, batch):
    """The targets of a mini-batch (the dict of memory.sample), a tensor (B,):
    y = r + DISCOUNT (1 - terminated) max_a' Q_frozen(s', a'), a constant for the gradient."""
    rewards = torch.as_tensor(batch["rewards"], dtype=torch.float32)
    alive = 1.0 - torch.as_tensor(batch["terminated"], dtype=torch.float32)
    follow = online(torch.as_tensor(batch["next_states"])).max(dim=1).values
    return rewards + DISCOUNT * alive * follow

In [ ]:
repaired_14_ok = False
with checked("B8.14"):
    repaired_14_ok = diagnose_14()

In [ ]:
# The colleague's DQN as it is now (before the repair: its symptoms; after: the repaired agent), STEPS_14 steps
with checked("B8.14"):
    start_14 = time.perf_counter()
    online_14, log_14 = run_colleague_14(0, STEPS_14)
    eval_14 = wbr.evaluate(wbr.greedy_q(online_14), 100, EVAL_SEED)
    fig_14, axes_14 = plt.subplots(1, 3, figsize=(11, 2.8))
    if log_14["returns"]:
        axes_14[0].plot(log_14["ends"], log_14["returns"], ".", alpha=0.3)
        plot_returns(axes_14[0], log_14["returns"], "moyenne sur 20", ends=log_14["ends"])
    axes_14[0].set(xlabel="pas", ylabel="retour", title="retours d'entraînement")
    axes_14[1].plot(np.arange(len(log_14["eps"])) * 100, log_14["eps"])
    axes_14[1].set(xlabel="pas", ylabel="ε", title="exploration", ylim=(0, 1.05))
    axes_14[2].plot(np.arange(1, len(log_14["q_mean"]) + 1) * 500, log_14["q_mean"], marker="o")
    axes_14[2].set(xlabel="pas", ylabel="moyenne de max Q", title="valeurs de 256 états de départ")
    plt.tight_layout()
    plt.show()
    print(f"{STEPS_14} pas en {time.perf_counter() - start_14:.0f} s, {len(log_14['returns'])} épisodes · politique "
          f"gloutonne sur 100 épisodes : {fr(eval_14['mean_return'], 1)} · max Q moyen à la fin : "
          f"{fr(log_14['q_mean'][-1], 1) if log_14['q_mean'] else '?'}")
    if repaired_14_ok:
        verdict("B8.14", eval_14["mean_return"] >= THRESHOLD_14,
                f"réparé, le DQN apprend : sa politique gloutonne dépasse {THRESHOLD_14:g}.",
                f"les quatre diagnostics passent, mais la politique gloutonne ne tient que {fr(eval_14['mean_return'], 1)} "
                f"pas (au moins {THRESHOLD_14:g} attendus) : as-tu changé autre chose que les quatre erreurs ?")
    else:
        print("⏳ Ex B8.14 : ces courbes montrent le DQN tel qu'il est ; la vérification ci-dessus dit ce qui reste.")

### Ex B8.15 — Ablations du DQN : sans réseau cible, sans rejeu, avec Double DQN — trois graines 🔬 ★★★ ⏱️ 50 min 🚀
**Objectif :** mesurer sur trois graines ce que le réseau cible, le rejeu et Double DQN changent à l'apprentissage, et rapporter la dispersion avec le résultat.  
**Prérequis :** Ex B8.13 · ✏️ B8.1 · fiche §208.2 (« La surestimation, et Double DQN »), §208.6 (« Les graines », « Rapporter plusieurs graines ») · **Fil rouge :** CartPole · **Parcours :** C

Quatre variantes du DQN de B8.13 : `"dqn"` ; `"no_target"`, **sans réseau cible** (les cibles viennent du réseau en ligne, toujours sans gradient) ; `"no_replay"`, **sans rejeu** (une mémoire de `batch_size` transitions : chaque mini-batch est fait des derniers pas, dans le désordre) ; `"double"`, **Double DQN**. **Contrat** des deux morceaux de la cellule suivante :
- `targets_15(online, frozen, next_states, rewards, terminated, gamma, variant)` rend les cibles d'un mini-batch pour une variante, un tenseur `(B,)` sans gradient : celles de `dqn_targets_13` avec le réseau cible `frozen` pour `"dqn"` et `"no_replay"` ; la même formule avec le réseau en ligne pour `"no_target"` ; pour `"double"`, l'action choisie par le réseau en ligne (`argmax`) et sa valeur lue dans `frozen` ;
- `memory_size_15(variant, settings=SETTINGS_13)` rend la capacité de la mémoire : `settings["capacity"]`, sauf `settings["batch_size"]` pour `"no_replay"`.

La vérification compare tes cibles à celles de la référence sur 6 transitions (deux réseaux qui ne choisissent pas toujours la même action), pour les quatre variantes. Puis l'expérience : en **mode rapide**, les courbes viennent d'un fichier précalculé (`wbr.precomputed("a15")`) : le DQN de référence du chapitre, 20 000 pas, trois graines par variante, la politique gloutonne évaluée tous les 1 000 pas sur 20 épisodes ; la cellule rejoue d'abord les 3 000 premiers pas d'une des courbes (Double DQN, première graine) pour vérifier qu'elle retombe sur le fichier. En **mode complet** 🚀 (`FAST_MODE = False`), elle relance tout avec tes deux morceaux dans la boucle de référence : 50 000 pas, trois graines par variante, environ 20 minutes de CPU. La figure trace la **médiane** des trois graines et leur **intervalle** (du minimum au maximum) ; un tableau donne la dernière évaluation de chaque graine.

Dans tes notes : quelle variante t'a surpris ? Les intervalles se recouvrent-ils, et que peux-tu conclure avec trois graines seulement (208.6, rliable) ? Pourquoi « sans rejeu » est-il ici une mémoire de 64 transitions plutôt qu'aucune mémoire, et que mesure-t-on alors ? Que faudrait-il pour affirmer qu'une variante est meilleure ?

In [ ]:
VARIANTS_15 = wbr.DQN_VARIANTS                 # ("dqn", "no_target", "no_replay", "double")
NAMES_15 = {"dqn": "DQN", "no_target": "sans réseau cible", "no_replay": "sans rejeu", "double": "Double DQN"}
A15 = wbr.precomputed("a15")                   # the curves of the reference: 4 variants x 3 seeds (protocol inside)
REPLAY_STEPS_15 = 3_000                        # the prefix of one recorded run that this notebook trains again

In [ ]:
def targets_15(online, frozen, next_states, rewards, terminated, gamma, variant):
    """The targets of a mini-batch for the four variants, a tensor (B,) without gradient: "dqn" and "no_replay", those of
    dqn_targets_13 with the target network `frozen`; "no_target", the same formula with the online network in place of
    the target network; "double", the action chosen by the online network (argmax), its value read in `frozen`."""
    raise NotImplementedError("targets_15")


def memory_size_15(variant, settings=SETTINGS_13):
    """The capacity of the replay memory of a variant: settings["capacity"], except for "no_replay":
    settings["batch_size"] (each mini-batch is then the last batch_size transitions, in a random order)."""
    raise NotImplementedError("memory_size_15")

In [ ]:
ok_15 = False
with checked("B8.15"):
    online_15, frozen_15 = wbr.q_network(32, seed=150), wbr.q_network(32, seed=151)
    next_15 = torch.as_tensor(np.random.default_rng(15).normal(0, 1, (6, 4)), dtype=torch.float32)
    rewards_15 = torch.tensor([1.0, 1.0, 0.5, 1.0, -1.0, 1.0])
    term_15 = torch.tensor([False, True, False, False, False, False])
    with torch.no_grad():
        disagree_15 = int((online_15(next_15).argmax(1) != frozen_15(next_15).argmax(1)).sum())
    good_15 = []
    for variant_15 in VARIANTS_15:
        y_15 = targets_15(online_15, frozen_15, next_15.clone(), rewards_15.clone(), term_15.clone(), 0.99, variant_15)
        if not returned("B8.15", f"targets_15(..., {variant_15!r})", y_15):
            good_15.append(False)
            continue
        if getattr(y_15, "requires_grad", False):
            print(f"❌ Ex B8.15 : targets_15(..., {variant_15!r}) porte un gradient (requires_grad=True).")
            good_15.append(False)
            continue
        good_15.append(same("B8.15", f"targets_15, variante {variant_15!r} (6 transitions, la 2ᵉ terminée)", y_15,
                            wbr.dqn_targets(online_15, frozen_15, next_15, rewards_15, term_15, 0.99, variant_15),
                            rtol=1e-6, atol=1e-6))
    sizes_15 = {v: memory_size_15(v) for v in VARIANTS_15}
    want_sizes_15 = {v: (SETTINGS_13["batch_size"] if v == "no_replay" else SETTINGS_13["capacity"]) for v in VARIANTS_15}
    good_15.append(sizes_15 == want_sizes_15)
    verdict("B8.15", good_15[-1], f"memory_size_15 : {sizes_15}.",
            f"memory_size_15 : attendu {want_sizes_15}, reçu {sizes_15}.")
    print(f"(le réseau en ligne et le réseau cible de la vérification choisissent une autre action dans "
          f"{disagree_15} des 6 états s′ : Double DQN y diffère de DQN)")
    ok_15 = all(good_15)

In [ ]:
# The four variants on three seeds (fast mode: the curves of the reference, precomputed; full mode: your pieces in the
# reference loop, 3 seeds x 4 variants, 🚀), then the median and the range of the three seeds
protocol_15 = A15["protocol"]
if FAST_MODE:
    runs_15 = A15["runs"]
    replay_15, replay_hist_15 = wbr.train_dqn(protocol_15["seeds"][0], protocol_15["steps"], variant="double",
                                              eval_every=protocol_15["eval_every"],
                                              eval_episodes=protocol_15["eval_episodes"],
                                              eval_seed=protocol_15["eval_seed"], stop_after=REPLAY_STEPS_15)
    recorded_15 = runs_15[f"double/{protocol_15['seeds'][0]}"]["eval_returns"][:len(replay_hist_15["eval_returns"])]
    if np.allclose(replay_hist_15["eval_returns"], recorded_15, atol=1e-6):
        print(f"✅ Ex B8.15 : Double DQN, graine {protocol_15['seeds'][0]}, rejoué sur {REPLAY_STEPS_15} pas : "
              f"{replay_hist_15['eval_returns']}, comme dans le fichier : les courbes ci-dessous sont celles de ce code.")
    else:
        print(f"ℹ️ Double DQN, graine {protocol_15['seeds'][0]}, rejoué sur {REPLAY_STEPS_15} pas : "
              f"{replay_hist_15['eval_returns']}, le fichier dit {recorded_15} : une autre machine ou une autre version "
              "de PyTorch arrondit autrement, et un écart minuscule change la suite d'un run de RL (208.6). Les courbes "
              "ci-dessous restent celles du fichier.")
elif not ok_15:
    runs_15 = None
    print("⏳ Ex B8.15 : le mode complet entraîne tes variantes ; il attend que la vérification ci-dessus passe.")
else:
    runs_15 = {}
    for variant_15 in VARIANTS_15:
        for seed_15 in protocol_15["seeds"]:
            _, h_15 = wbr.train_dqn(seed_15, wbr.dqn_steps(False), variant=variant_15, eval_every=2_500,
                                    eval_episodes=protocol_15["eval_episodes"], eval_seed=protocol_15["eval_seed"],
                                    targets_fn=targets_15, capacity=memory_size_15(variant_15))
            runs_15[f"{variant_15}/{seed_15}"] = h_15
            print(f"{NAMES_15[variant_15]}, graine {seed_15} : {h_15['seconds']:.0f} s")
if runs_15:
    fig_15, ax_15 = plt.subplots(figsize=(8, 3.6))
    finals_15 = {}
    for variant_15 in VARIANTS_15:
        hs_15 = [runs_15[f"{variant_15}/{s}"] for s in protocol_15["seeds"]]
        steps_15 = hs_15[0]["eval_steps"]
        b_15 = wbr.band([h["eval_returns"] for h in hs_15])
        line_15, = ax_15.plot(steps_15, b_15["median"], label=NAMES_15[variant_15])
        ax_15.fill_between(steps_15, b_15["low"], b_15["high"], color=line_15.get_color(), alpha=0.15)
        finals_15[variant_15] = [h["eval_returns"][-1] for h in hs_15]
    ax_15.axhline(RANDOM_LEVEL, color="gray", ls=":", lw=1)
    ax_15.set(xlabel="pas d'environnement", ylabel="retour glouton (20 épisodes)",
              title="Médiane et intervalle (min-max) sur trois graines")
    ax_15.legend(fontsize=8)
    plt.show()
    for variant_15, values_15 in finals_15.items():
        print(f"{NAMES_15[variant_15]:18s} dernière évaluation par graine : {[round(v, 1) for v in values_15]} · "
              f"médiane {fr(float(np.median(values_15)), 1)}")

## Partie B · La politique : REINFORCE, GAE et PPO

Tu changes de famille : au lieu d'apprendre $Q$ et d'en déduire la politique, tu ajustes **directement** les probabilités des actions. Tu prédis d'abord ce que change une baseline à REINFORCE (B8.16), tu le mesures sur le gradient et sur l'apprentissage (B8.17), tu écris dans ta librairie les deux calculs que les implémentations ratent le plus souvent, GAE et l'objectif clippé (B8.18, B8.19), puis tu assembles ton PPO sur des environnements vectorisés (B8.20). La suite de la partie lit ses journaux, règle ses hyperparamètres et teste sa robustesse.

### Ex B8.16 — Prédire : REINFORCE sans baseline, avec baseline, avec retours normalisés — lequel apprend le plus vite ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire, avant de les voir, ce que la baseline et la normalisation des retours changent à REINFORCE sur CartPole.  
**Prérequis :** ∂ B8.7 · 🧠 Q6 · fiche §208.3 (« REINFORCE », « La baseline et la variance ») · **Fil rouge :** CartPole · **Parcours :** C

Trois variantes de REINFORCE, une mise à jour (un pas d'Adam, learning rate $10^{-2}$) par épisode, $\gamma = 0{,}99$, la politique 4 → 64 → 2 (tanh), 300 épisodes : **sans baseline** (le poids de $\log\pi(a_t \mid s_t)$ est $G_t$) ; avec une **baseline moyenne** ($G_t - b$, $b$ la moyenne des $G_t$ des 20 épisodes précédents, 0 au premier) ; avec des **retours normalisés** ($(G_t - \bar G)/\sigma_G$ sur l'épisode). Chacune a été entraînée avec six graines (le fichier `wbr.precomputed("p16")`, lu par l'expérience). La fiche dit seulement qu'une baseline ne biaise pas le gradient et réduit sa variance sur un **bandit jouet** ; ici, c'est CartPole. **Avant d'exécuter**, écris tes trois prédictions :
- a) `best_16` : laquelle des trois variantes a la plus grande **médiane**, sur les six graines, du retour moyen de ses 50 derniers épisodes ? `0` sans baseline, `1` la baseline moyenne, `2` les retours normalisés ;
- b) `first_200_16` : sur combien des six graines la variante **sans baseline** atteint-elle, au moins une fois, une moyenne glissante de 200 (sur 20 épisodes) en 300 épisodes ? Un entier de 0 à 6 ;
- c) `beats_16` : la baseline moyenne finit-elle **au-dessus** de la variante sans baseline (50 derniers épisodes) sur **chacune** des six graines ? `True` ou `False`.

La cellule d'expérience n'affiche rien tant que tes trois prédictions ne sont pas écrites ; elle trace ensuite la médiane et l'intervalle des six graines pour chaque variante, donne les valeurs par graine et vérifie tes prédictions. Rien n'y dépend de ton code.

Dans tes notes, avant d'exécuter : sur CartPole, tous les retours $G_t$ sont positifs ; que fait alors REINFORCE sans baseline à **chaque** action jouée ? Qu'est-ce qui distingue la normalisation d'une simple baseline ?

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
# Your predictions, BEFORE running the next cell (an int, or True/False)
best_16 = ...        # a) 0 (no baseline), 1 (mean baseline) or 2 (normalised returns)
first_200_16 = ...   # b) an int from 0 to 6
beats_16 = ...       # c) True or False

In [ ]:
if any(answer is ... or answer is None for answer in [best_16, first_200_16, beats_16]):
    print("⏳ Ex B8.16 : écris d'abord tes trois prédictions (best_16, first_200_16, beats_16) dans la cellule précédente.")
else:
    P16 = wbr.precomputed("p16")                  # REINFORCE, 3 variants x 6 seeds x 300 episodes (protocol inside)
    names_16 = {"none": "sans baseline", "mean": "baseline moyenne", "normalize": "retours normalisés"}
    seeds_16 = P16["protocol"]["seeds"]
    fig_16, ax_16 = plt.subplots(figsize=(8, 3.4))
    last50_16 = {}
    for baseline_16, name_16 in names_16.items():
        curves_16 = [wbr.moving_average(P16["runs"][f"{baseline_16}/{s}"]["episode_returns"], 20) for s in seeds_16]
        b_16 = wbr.band(curves_16)
        line_16, = ax_16.plot(np.arange(1, 301), b_16["median"], label=name_16)
        ax_16.fill_between(np.arange(1, 301), b_16["low"], b_16["high"], color=line_16.get_color(), alpha=0.12)
        last50_16[baseline_16] = [float(np.mean(P16["runs"][f"{baseline_16}/{s}"]["episode_returns"][-50:])) for s in seeds_16]
    ax_16.axhline(200, color="gray", ls=":", lw=1)
    ax_16.set(xlabel="épisode", ylabel="retour (moyenne glissante sur 20)",
              title=f"REINFORCE sur CartPole : médiane et intervalle sur {len(seeds_16)} graines")
    ax_16.legend(fontsize=8)
    plt.show()
    for baseline_16, name_16 in names_16.items():
        reached_16 = sum(bool((wbr.moving_average(P16["runs"][f"{baseline_16}/{s}"]["episode_returns"], 20) >= 200).any())
                         for s in seeds_16)
        print(f"{name_16:20s} moyenne des 50 derniers épisodes, par graine : {[round(v) for v in last50_16[baseline_16]]} · "
              f"médiane {fr(float(np.median(last50_16[baseline_16])), 1)} · moyenne glissante ≥ 200 atteinte sur "
              f"{reached_16} graines sur {len(seeds_16)}")
    wb.check("B8.16a", best_16)
    wb.check("B8.16b", first_200_16)
    wb.check("B8.16c", beats_16)

Dans tes notes, après l'expérience : la dispersion d'une graine à l'autre est-elle plus grande pour une variante que pour les autres ? Une seule graine aurait-elle suffi à répondre à a) ? Qu'est-ce qui, dans la figure, relève de la **variance** du gradient et qu'est-ce qui relève de l'**échelle** des pas ?

### Ex B8.17 — REINFORCE avec et sans baseline : variance du gradient et vitesse d'apprentissage 📦 ★★★ ⏱️ 45 min
**Objectif :** écrire REINFORCE en PyTorch, mesurer la dispersion de son gradient avec et sans baseline, et relier cette dispersion à la vitesse d'apprentissage.  
**Prérequis :** 🔮 B8.16 · 🔨 26.14 (`returns_to_go`) · ∂ B8.7 · fiche §208.3 · **Fil rouge :** CartPole · **Parcours :** C

**Contrat** des trois fonctions de la cellule suivante :
- `make_policy_17(hidden=64)` rend un `nn.Module` neuf : 4 entrées, une couche cachée de `hidden` neurones avec tanh, 2 sorties (les logits des deux actions) ;
- `episode_gradients_17(policy_net, episodes, baseline, gamma=0.99)` rend un tableau NumPy float64 de forme `(M, P)` : la ligne $i$ est le gradient, par rapport à **tous** les paramètres de `policy_net` mis bout à bout dans l'ordre de `policy_net.parameters()` (`P` nombres), de la loss de l'épisode $i$, $-\frac{1}{T}\sum_t \log\pi(a_t \mid s_t)\,(G_t - b)$. `episodes` est une liste de $M$ dictionnaires `"states"` (float32 `(T, 4)`), `"actions"` (int64 `(T,)`) et `"rewards"` (`(T,)`) ; $G_t$ vient de `mylearn.rl.returns_to_go` (ta fonction de 26.14) ; $b = 0$ pour `"none"`, et pour `"mean"` la moyenne des $G_t$ de **tous** les épisodes donnés (une seule constante) ;
- `train_reinforce_17(seed, n_episodes, baseline, lr=1e-2, gamma=0.99)` entraîne REINFORCE sur CartPole-v1 (`wbr.make_env()`) et rend le couple `(policy_net, returns)`, `returns` la liste des retours des `n_episodes` épisodes d'entraînement : `torch.manual_seed(seed)` puis `make_policy_17()` ; `env.reset(seed=seed)` une fois au départ ; chaque épisode joué en entier avec des actions **tirées** dans la politique (`torch.distributions.Categorical`) ; puis un pas d'Adam (`lr`) sur la loss de l'épisode, avec $b = 0$ (`"none"`) ou $b$ la moyenne des $G_t$ des 20 épisodes **précédents** (`"mean"`, 0 au premier épisode).

La cellule donnée fixe deux politiques (une non entraînée ; une entraînée 60 épisodes par la référence, ses poids relus) et 32 épisodes joués par chacune (`EPISODES_17`). La vérification compare tes gradients par épisode à ceux de la référence, pour les deux politiques et les deux baselines, puis affiche leur **dispersion**, $\frac{1}{M}\sum_i \lVert g_i - \bar g\rVert^2$ (`wbr.gradient_spread`), avec et sans baseline. La cellule suivante entraîne ta REINFORCE sans puis avec baseline (300 épisodes chacune, environ une minute : les épisodes longs coûtent) et trace tes courbes sur l'intervalle des six graines de la référence (le protocole de B8.16) ; elle attend qu'avec la baseline, tes 50 derniers épisodes dépassent `THRESHOLD_17`.

Dans tes notes : de combien la baseline réduit-elle la dispersion pour chacune des deux politiques, et pourquoi si peu pour l'une d'elles (pense à la longueur des épisodes et à la taille des $G_t$) ? Une baseline **constante** peut-elle augmenter la variance ? Le gradient moyen change-t-il avec la baseline (∂ B8.7), et le vois-tu sur 32 épisodes ? Ta courbe tombe-t-elle dans l'intervalle de la référence ?

In [ ]:
C17 = wbr.precomputed("c17")                  # a policy trained 60 episodes by the reference (its weights)
HALF_TRAINED_17 = wbr.policy_network(64)
HALF_TRAINED_17.load_state_dict({k: torch.tensor(v) for k, v in C17["state_dict"].items()})
POLICIES_17 = {"non entraînée": wbr.policy_network(64, seed=17), "entraînée 60 épisodes": HALF_TRAINED_17}
EPISODES_17 = {name: wbr.collect_episodes(net, 32, 1700) for name, net in POLICIES_17.items()}   # 32 episodes each
N_17 = 300                                       # training episodes of each variant (B8.16 shows the same protocol)
SEED_17 = 17
THRESHOLD_17 = 100                    # the last 50 episodes with the mean baseline (6 seeds of the reference)
for name_17, eps_17 in EPISODES_17.items():
    print(f"politique {name_17} : 32 épisodes, {fr(np.mean([len(e['rewards']) for e in eps_17]), 1)} pas en moyenne")

In [ ]:
def make_policy_17(hidden=64):
    """A new policy network for CartPole: 4 inputs, one hidden layer of `hidden` neurons with tanh, 2 outputs (the
    logits of the two actions)."""
    raise NotImplementedError("make_policy_17")


def episode_gradients_17(policy_net, episodes, baseline, gamma=0.99):
    """One REINFORCE gradient per episode (the contract of the statement): a float64 array (M, P), row i the gradient
    of -mean_t(log pi(a_t | s_t) (G_t - b)) of episode i with respect to all the parameters of policy_net, flattened
    in the order of policy_net.parameters(); b = 0 ("none") or the mean of the G_t of ALL the episodes ("mean")."""
    raise NotImplementedError("episode_gradients_17")


def train_reinforce_17(seed, n_episodes, baseline, lr=1e-2, gamma=0.99):
    """REINFORCE on CartPole-v1 (the contract of the statement): one Adam step per episode; baseline "none" or "mean"
    (the mean of the G_t of the 20 previous episodes, 0 for the first). Returns (policy_net, returns), the return of
    each training episode."""
    raise NotImplementedError("train_reinforce_17")

In [ ]:
grads_ok_17 = False
with checked("B8.17"):
    net_17 = make_policy_17()
    if returned("B8.17", "make_policy_17", net_17):
        out_17 = net_17(torch.zeros(3, 4)) if isinstance(net_17, nn.Module) else None
        verdict("B8.17", out_17 is not None and tuple(out_17.shape) == (3, 2),
                f"make_policy_17 : {sum(p.numel() for p in net_17.parameters())} paramètres, deux logits par observation.",
                "make_policy_17 : attendu un nn.Module qui rend (3, 2) pour 3 observations.")

with checked("B8.17"):
    if rl_ready("B8.17", "returns_to_go"):
        for name_17, net_17 in POLICIES_17.items():
            for baseline_17 in ("none", "mean"):
                copy_17 = copy.deepcopy(net_17)
                got_17 = episode_gradients_17(copy_17, EPISODES_17[name_17], baseline_17)
                want_17 = wbr.episode_gradients(net_17, EPISODES_17[name_17], baseline_17)
                if not returned("B8.17", "episode_gradients_17", got_17):
                    raise StopCheck
                arr_17 = np.asarray(got_17.detach() if isinstance(got_17, torch.Tensor) else got_17, dtype=float)
                label_17 = f"episode_gradients_17, politique {name_17}, baseline {baseline_17!r}"
                lengths_17 = np.array([len(e["rewards"]) for e in EPISODES_17[name_17]], dtype=float)
                near_17 = lambda a, b: a.shape == b.shape and np.linalg.norm(a - b) <= 1e-3 * np.linalg.norm(b)
                if near_17(arr_17, -want_17):
                    print(f"❌ Ex B8.17 : {label_17} : tous les signes sont inversés : on minimise la loss −mean(log π · (G − b)).")
                    raise StopCheck
                if near_17(arr_17, want_17 * lengths_17[:, None]):
                    print(f"❌ Ex B8.17 : {label_17} : chaque ligne vaut T fois l'attendu (T, la longueur de son épisode) : "
                          "une somme sur les pas au lieu d'une moyenne ?")
                    raise StopCheck
                if baseline_17 == "mean" and near_17(arr_17, np.concatenate(
                        [wbr.episode_gradients(net_17, [e], "mean") for e in EPISODES_17[name_17]])):
                    print(f"❌ Ex B8.17 : {label_17} : ta baseline est la moyenne des G_t de chaque épisode ; le contrat "
                          "demande une seule constante, la moyenne des G_t de tous les épisodes donnés.")
                    raise StopCheck
                if near_17(arr_17, wbr.episode_gradients(net_17, EPISODES_17[name_17], baseline_17, gamma=1.0)):
                    print(f"❌ Ex B8.17 : {label_17} : ce sont les gradients avec γ = 1 : tes retours sont-ils actualisés "
                          "avec gamma ?")
                    raise StopCheck
                if not same("B8.17", label_17, arr_17, want_17, rtol=1e-4, atol=1e-6):
                    raise StopCheck
        grads_ok_17 = True
        if grads_ok_17:
            for name_17, net_17 in POLICIES_17.items():
                spread_17 = {b: wbr.gradient_spread(episode_gradients_17(copy.deepcopy(net_17), EPISODES_17[name_17], b))
                             for b in ("none", "mean")}
                print(f"politique {name_17} : dispersion des gradients (moyenne de ‖g_i − ḡ‖²) sans baseline "
                      f"{spread_17['none']:.3g}, avec {spread_17['mean']:.3g}, rapport {fr(spread_17['none'] / spread_17['mean'], 2)}")

In [ ]:
# Training: REINFORCE without and with the mean baseline, N_17 episodes each, against the reference curves of B8.16
if not grads_ok_17:
    print("⏳ Ex B8.17 : l'entraînement attend que tes gradients par épisode passent la vérification ci-dessus.")
else:
    with checked("B8.17"):
        P16_17 = wbr.precomputed("p16")
        runs_17 = {}
        for baseline_17 in ("none", "mean"):
            start_17 = time.perf_counter()
            net_17, returns_17 = pair("B8.17", "train_reinforce_17", train_reinforce_17(SEED_17, N_17, baseline_17),
                                      names=("policy_net", "returns"))
            runs_17[baseline_17] = (net_17, np.asarray(returns_17, dtype=float), time.perf_counter() - start_17)
        fig_17, axes_17 = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
        for ax_17, baseline_17 in zip(axes_17, ("none", "mean")):
            ref_17 = wbr.band([wbr.moving_average(P16_17["runs"][f"{baseline_17}/{s}"]["episode_returns"], 20)
                               for s in P16_17["protocol"]["seeds"]])
            ax_17.fill_between(np.arange(1, 301), ref_17["low"], ref_17["high"], color="gray", alpha=0.2,
                               label="référence, 6 graines (min-max)")
            plot_returns(ax_17, runs_17[baseline_17][1], f"toi, graine {SEED_17}")
            ax_17.set(xlabel="épisode", title={"none": "sans baseline", "mean": "baseline moyenne"}[baseline_17])
        axes_17[0].set_ylabel("retour (moyenne glissante sur 20)")
        axes_17[0].legend(fontsize=8)
        plt.show()
        for baseline_17, (net_17, returns_17, seconds_17) in runs_17.items():
            ev_17 = wbr.evaluate(wbr.greedy_logits(net_17), 100, EVAL_SEED)
            print(f"{baseline_17:5s} : {len(returns_17)} épisodes en {seconds_17:.0f} s · 50 derniers : "
                  f"{fr(returns_17[-50:].mean(), 1)} · politique déterministe sur 100 épisodes : {fr(ev_17['mean_return'], 1)}")
        mean_17 = runs_17["mean"][1]
        verdict("B8.17", len(mean_17) == N_17 and mean_17[-50:].mean() >= THRESHOLD_17,
                f"avec la baseline, tes 50 derniers épisodes dépassent {THRESHOLD_17:g} (la pire des 6 graines de la "
                f"référence fait 203).",
                f"avec la baseline, attendu {N_17} épisodes dont les 50 derniers dépassent {THRESHOLD_17:g} en moyenne, "
                f"reçu {len(mean_17)} épisodes et {fr(float(mean_17[-50:].mean()), 1) if len(mean_17) else '?'} ; si ta "
                "courbe monte puis retombe vers 9 (une politique devenue déterministe, qui pousse toujours du même côté), "
                "relance avec une autre SEED_17 avant de chercher une erreur : REINFORCE s'effondre parfois.")

### Ex B8.18 — compute_gae et explained_variance 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire GAE à rebours, avec la distinction entre fin d'épisode et limite de temps, et la variance expliquée d'un critique, en NumPy.  
**Prérequis :** ✏️ B8.4 · fiche §208.4 (« GAE », « Fin d'épisode ou limite de temps », « Le critique et sa variance expliquée ») · **mylearn :** `deep_rl.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/deep_rl.py` (créé par `python tools/start_chapter.py B8`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement, sans PyTorch. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb8_deep_rl.py -q` les lance tous dans un terminal.

**Contrat** de `compute_gae(rewards, values, next_values, terminated, truncated, gamma=0.99, lam=0.95)`, qui rend le couple `(advantages, returns)` :
- les cinq tableaux ont la même forme, `(T,)` ou `(T, n_envs)`, le **temps sur le premier axe** ; chaque colonne est un rollout à part, avec sa propre récursion ;
- $\delta_t = r_t + \gamma(1 - \text{terminated}_t)\,\text{next\_values}_t - \text{values}_t$, puis à rebours $\hat A_t = \delta_t + \gamma\lambda(1 - \text{done}_t)\,\hat A_{t+1}$, avec $\text{done}_t$ = `terminated` **ou** `truncated` et $\hat A_T = 0$ après le dernier pas ;
- `next_values[t]` est **toujours** la valeur de l'observation qui suit le pas $t$, au dernier pas du rollout compris : il n'y a pas de valeur d'amorçage à part, et on ne la reprend pas dans `values[t+1]` ; un pas tronqué amorce sur sa `next_values` (la vraie dernière observation) mais arrête la récursion ; `terminated` et `truncated` peuvent être vrais ensemble ;
- `returns` vaut `advantages + values` (pas `next_values`), de la forme de `rewards` ;
- $\gamma$ et $\lambda$ dans $[0, 1]$, bornes comprises ; sinon `ValueError`.

**Contrat** de `explained_variance(y_pred, y_true)`, qui rend un `float` : $1 - \operatorname{Var}(y_{\text{true}} - y_{\text{pred}})/\operatorname{Var}(y_{\text{true}})$, **la prédiction d'abord** (l'ordre inverse de `sklearn.metrics.explained_variance_score`), des variances de population (`np.var`) ; `nan` **seulement** quand la variance de `y_true` vaut exactement 0 ; une prédiction constante donne 0, pas `nan`. Attention : `np.var([0.1, 0.1, 0.1])` vaut $1{,}9 \times 10^{-34}$ et non 0 ; les tests n'emploient que des constantes exactes en binaire (3, 0, −2,5), et acceptent aussi une constante détectée autrement (`np.ptp(y_true) == 0`).

**Pour les deux** : les entrées ne sont jamais modifiées ; des formes différentes lèvent `ValueError`, même quand NumPy pourrait les diffuser (`(T, 1)` contre `(T,)`, un tableau de longueur 1), y compris pour `terminated` et `truncated` ; des listes et des tableaux float32 sont acceptés ; les tests comparent à $10^{-6}$ près.

La première cellule de vérification essaie tes fonctions sur les deux exemples de la docstring, le tableau de 208.4, un rollout `(9, 3)` comparé colonne par colonne et les trois exemples de variance expliquée de 208.4, puis affiche ce que rend ta `compute_gae` sur le rollout de ✏️ B8.4 (compare à tes réponses). La seconde lance les 79 tests de l'exercice (`-k "test_compute_gae_ or test_explained_variance_"`).

Dans tes notes : que donnent λ = 0 et λ = 1 (lis-le sur ta fonction) ? Pourquoi `next_values` plutôt que `values` décalé d'un pas ? Que se passerait-il au dernier pas d'un rollout si l'on prenait 0 pour la valeur suivante ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B8")   # reload your saved file
with checked("B8.18"):
    r_18, v_18 = [1.0, 1.0, 1.0], [0.0, 0.0, 0.0]
    out_18 = mylearn.deep_rl.compute_gae(r_18, v_18, v_18, [False, False, True], [False, False, False], gamma=0.5,
                                         lam=1.0)
    adv_18, ret_18 = pair("B8.18", "compute_gae", out_18, names=("advantages", "returns"))
    same("B8.18", "exemple 1 de la docstring (avantages)", adv_18, [1.75, 1.5, 1.0])
    adv_18, _ = pair("B8.18", "compute_gae", mylearn.deep_rl.compute_gae(
        r_18, [0.5, 0.5, 0.5], [0.5, 0.5, 2.0], [False, False, False], [False, True, False], gamma=0.9, lam=0.5),
        names=("advantages", "returns"))
    same("B8.18", "exemple 2 de la docstring (troncature au pas 1)", adv_18, [1.3775, 0.95, 2.3])
    v_fiche_18 = [2.0, 1.5, 0.8]
    adv_18, ret_18 = pair("B8.18", "compute_gae", mylearn.deep_rl.compute_gae(
        r_18, v_fiche_18, [1.5, 0.8, 0.0], [False, False, True], [False, False, False], gamma=0.9, lam=0.5),
        names=("advantages", "returns"))
    same("B8.18", "le tableau de 208.4 (avantages)", adv_18, [0.4895, 0.31, 0.2])
    same("B8.18", "le tableau de 208.4 (cibles du critique)", ret_18, [2.4895, 1.81, 1.0])
    rng_18 = np.random.default_rng(18)
    cols_18 = [rng_18.normal(size=(9, 3)) for _ in range(3)] + [rng_18.random((9, 3)) < 0.2, rng_18.random((9, 3)) < 0.2]
    passed_18 = [c.copy() for c in cols_18]
    adv2_18, _ = pair("B8.18", "compute_gae", mylearn.deep_rl.compute_gae(*passed_18, 0.95, 0.9),
                      names=("advantages", "returns"))
    verdict("B8.18", all(np.array_equal(a, b) for a, b in zip(passed_18, cols_18)), "compute_gae laisse ses entrées intactes.",
            "compute_gae a modifié un de ses tableaux d'entrée : travaille sur des copies.")
    one_18 = np.stack([wbr.gae_numpy(*[c[:, j] for c in cols_18], 0.95, 0.9)[0] for j in range(3)], axis=1)
    same("B8.18", "un rollout (9, 3) : chaque colonne comme un rollout à elle seule", adv2_18, one_18, rtol=1e-6, atol=1e-6)
    ev_18 = mylearn.deep_rl.explained_variance([1.5, 1.5, 3.5, 3.5], [1.0, 2.0, 3.0, 4.0])
    same("B8.18", "variance expliquée de 208.4 (prédictions d'abord)", ev_18, 0.8)
    same("B8.18", "une prédiction décalée de 5 partout", mylearn.deep_rl.explained_variance([6.0, 7.0, 8.0, 9.0],
                                                                                         [1.0, 2.0, 3.0, 4.0]), 1.0)
    same("B8.18", "des retours tous égaux", mylearn.deep_rl.explained_variance([1.0, 2.0], [3.0, 3.0]), np.nan)
    b84_18 = mylearn.deep_rl.compute_gae([1.0] * 4, [4.0, 3.8, 2.0, 2.4], [3.8, 3.6, 2.4, 2.6], [False] * 4,
                                         [False, True, False, False], gamma=0.9, lam=0.95)[0]
    print("le rollout de ✏️ B8.4 (λ = 0,95) avec ta fonction :", np.round(np.asarray(b84_18, dtype=float), 4).tolist(),
          "← compare à tes réponses de B8.4 d) et e)")

In [ ]:
if written(lambda: mylearn.deep_rl.compute_gae([1.0], [0.0], [0.0], [True], [False])):
    run_pytest_group(TEST_DEEP_RL, "test_compute_gae_ or test_explained_variance_")   # 79 tests
else:
    print("⏳ Ex B8.18 : les tests attendent tes fonctions compute_gae et explained_variance.")

### Ex B8.19 — clipped_surrogate : valeur et gradient par rapport au ratio 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire l'objectif clippé de PPO et son gradient par rapport aux ratios, cas par cas, sans autograd.  
**Prérequis :** ✏️ B8.5 · fiche §208.5 (« L'objectif clippé ») · **mylearn :** `deep_rl.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/deep_rl.py` (créé par `python tools/start_chapter.py B8`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement, sans PyTorch. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb8_deep_rl.py -q` les lance tous dans un terminal.

**Contrat** de `clipped_surrogate(ratio, advantages, clip_eps=0.2)`, qui rend le couple `(loss, gradient)` :
- `loss` $= -\frac{1}{n}\sum_i \min\big(\rho_i \hat A_i,\ \operatorname{clip}(\rho_i, 1 - \varepsilon, 1 + \varepsilon)\,\hat A_i\big)$, un `float` (la convention de la fiche : on minimise l'opposé de $L^{\text{CLIP}}$) ;
- `gradient` est la dérivée de `loss` par rapport à chaque ratio, un tableau `(n,)` : $-\hat A_i/n$ quand le terme non clippé est le minimum, **égalités comprises** (un ratio dans l'intervalle, ou **sur un bord exact**, comme l'autograd de `torch.clamp`), et 0 quand le terme clippé est **strictement** plus petit ; un avantage nul donne un gradient nul. Une façon de vérifier : pour $\hat A > 0$, le gradient est non nul tant que $\rho \le 1 + \varepsilon$ ; pour $\hat A < 0$, tant que $\rho \ge 1 - \varepsilon$ ;
- `ratio` et `advantages` de forme `(n,)`, sans diffusion (des formes différentes lèvent `ValueError`) ; un ratio de 0 est accepté, un ratio négatif lève `ValueError` ; `clip_eps` doit être > 0 (sinon `ValueError`) et **peut dépasser 1** ;
- les entrées ne sont jamais modifiées ; listes et tableaux float32 acceptés ; tolérance $10^{-6}$.

La première cellule de vérification essaie ta fonction sur l'exemple de 208.5, puis la compare à l'autograd de PyTorch (`-torch.min(r * A, r.clamp(1 - eps, 1 + eps) * A).mean()`) sur 9 ratios, dont deux sur un bord exact, pour ε = 0,2, 0,5 et 1,5. La seconde lance les 38 tests de l'exercice (`-k "test_clipped_surrogate_"`).

Dans tes notes : dans quelles zones de la figure 1 de l'article le gradient s'annule-t-il ? Pourquoi un ratio monté au-dessus de $1 + \varepsilon$ avec un avantage **négatif** garde-t-il son gradient ? Que devient l'objectif avec $\varepsilon \ge 1$ du côté bas ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B8")   # reload your saved file
with checked("B8.19"):
    loss_19, grad_19 = pair("B8.19", "clipped_surrogate", mylearn.deep_rl.clipped_surrogate([0.7, 1.1, 1.35],
                                                                                           [-1.0, 2.0, -1.0], 0.2),
                            names=("loss", "gradient"), floats=("loss",))
    same("B8.19", "l'exemple de 208.5 (loss)", loss_19, -(-0.8 + 2.2 - 1.35) / 3)
    same("B8.19", "l'exemple de 208.5 (gradient)", grad_19, [0.0, -2 / 3, 1 / 3])
    rng_19 = np.random.default_rng(19)
    for eps_19 in (0.2, 0.5, 1.5):
        ratio_19 = np.concatenate([rng_19.uniform(0.0, 2.5, 6), [1 - eps_19 if eps_19 < 1 else 0.0, 1 + eps_19, 1.0]])
        adv_19 = np.concatenate([rng_19.normal(size=6), [1.5, -1.5, 0.0]])
        r_t = torch.tensor(ratio_19, requires_grad=True)
        a_t = torch.tensor(adv_19)
        want_19 = -torch.min(r_t * a_t, r_t.clamp(1 - eps_19, 1 + eps_19) * a_t).mean()
        want_19.backward()
        loss_19, grad_19 = pair("B8.19", "clipped_surrogate",
                                mylearn.deep_rl.clipped_surrogate(ratio_19.copy(), adv_19.copy(), eps_19),
                                names=("loss", "gradient"), floats=("loss",))
        same("B8.19", f"9 ratios (deux sur un bord exact), ε = {fr(eps_19, 1)} : la loss contre l'autograd", loss_19,
             want_19.item(), rtol=1e-9, atol=1e-12)
        same("B8.19", f"9 ratios, ε = {fr(eps_19, 1)} : le gradient contre l'autograd", grad_19, r_t.grad.numpy(),
             rtol=1e-9, atol=1e-12)

In [ ]:
if written(lambda: mylearn.deep_rl.clipped_surrogate([1.0], [1.0])):
    run_pytest_group(TEST_DEEP_RL, "test_clipped_surrogate_")   # 38 tests
else:
    print("⏳ Ex B8.19 : les tests attendent ta fonction clipped_surrogate.")

### Ex B8.20 — PPO from scratch sur CartPole : environnements vectorisés, acteur-critique, GAE, clip, entropie 📦 ★★★★ ⏱️ 150 min
**Objectif :** assembler un PPO complet sur des environnements vectorisés, avec ta GAE et ton objectif clippé, et le voir résoudre CartPole.  
**Prérequis :** Ex B8.17, 🔨 B8.18, B8.19 · ✏️ B8.4, B8.5 · fiche §208.4, §208.5 (« La loss complète et la boucle », « Les environnements vectorisés ») · 🔁 R2 · **Fil rouge :** CartPole · **Parcours :** C

**Contrat** des quatre morceaux de la cellule suivante (les réglages sont dans `SETTINGS_20`, ceux du PPO de référence du chapitre, affichés par la cellule donnée) :
- `ActorCritic20(hidden=64)`, un `nn.Module` : un **acteur** (4 → `hidden` → `hidden` → 2 logits, tanh) et un **critique** (4 → `hidden` → `hidden` → 1, tanh), deux réseaux séparés dans les attributs `actor` et `critic` ; `forward(obs)` rend les logits `(N, 2)`, `value(obs)` rend $V$ de forme `(N,)` ;
- `rollout_20(agent, envs, obs, just_reset, n_steps, generator)` joue `n_steps` pas dans chacune des $N$ copies de `envs` (un `gym.make_vec` en autoreset next-step) et rend le triplet `(batch, obs, just_reset)` :
  - les actions sont **tirées** dans la politique de l'agent, sans gradient (avec `generator`, un `torch.Generator`) ;
  - `obs` `(N, 4)` est l'observation courante ; `just_reset` `(N,)` vaut `True` pour une copie dont le pas précédent a fini l'épisode : son prochain pas ne fait que le reset ;
  - `batch` est un dictionnaire de tableaux NumPy de forme `(n_steps, N, ...)` : `"obs"` (avant le pas), `"actions"`, `"log_probs"` (de l'action jouée), `"values"` ($V$ de `obs`), `"rewards"`, `"terminated"`, `"truncated"`, `"next_obs"` (ce que rend le pas), `"next_values"` ($V$ de `next_obs` : au pas où un épisode finit, celle de la **vraie dernière observation**) et `"valid"` (`False` au pas qui ne fait que le reset d'une copie : ce n'est pas une transition) ;
  - les `obs` et `just_reset` rendus servent au rollout suivant ;
- `ppo_loss_20(agent, obs, actions, old_log_probs, advantages, returns, settings=SETTINGS_20)` rend le couple `(loss, parts)` pour un mini-batch de tenseurs : `loss` $= L^{\pi} + c_1 L^V - c_2 H$ (un tenseur, avec son graphe), $L^{\pi} = -\operatorname{mean}\min(\rho \hat A, \operatorname{clip}(\rho, 1 \pm \varepsilon)\hat A)$, $\rho = \exp(\log\pi(a \mid s) - \text{old\_log\_probs})$, $L^V = \frac12 \operatorname{mean}(V(s) - \text{returns})^2$, $H$ l'entropie moyenne de $\pi(\cdot \mid s)$, $c_1$ = `vf_coef`, $c_2$ = `ent_coef`, $\varepsilon$ = `clip_eps` ; les avantages servent tels quels (on les normalise avant) ; `parts` est un dictionnaire de `float` : `"policy_loss"`, `"value_loss"`, `"entropy"`, `"approx_kl"` (la moyenne de $(\rho - 1) - \ln\rho$), `"clipfrac"` (la part des $|\rho - 1| > \varepsilon$) ;
- `train_ppo_20(seed, total_steps, settings=SETTINGS_20, reward_fn=None)` entraîne PPO et rend le couple `(agent, logs)` :
  - **départ** : `torch.manual_seed(seed)`, l'agent, Adam (`lr`, `eps=adam_eps`), un `torch.Generator().manual_seed(seed)` pour les actions et les mélanges, `n_envs` copies de CartPole-v1 (`gym.make_vec(..., vectorization_mode="sync")`, autoreset next-step) remises à zéro une fois avec `seed` ;
  - **les itérations** : `total_steps // (n_envs × n_steps)` ; à chacune, le learning rate `lr × (1 − it/n_iter)` quand `anneal_lr`, un rollout, puis les avantages et les retours par `mylearn.deep_rl.compute_gae` (ta fonction de B8.18, **appelée par ce nom** : la vérification l'observe) avec `gamma` et `gae_lambda`, et la variance expliquée par `mylearn.deep_rl.explained_variance`, sur les transitions valides ;
  - **la mise à jour** : `n_epochs` passages sur les transitions **valides**, mélangées puis coupées en `n_minibatches` mini-batches ; dans chacun, les avantages normalisés (`norm_adv`), `ppo_loss_20`, `clip_grad_norm_(..., max_grad_norm)` et un pas d'Adam ; et `mylearn.deep_rl.clipped_surrogate` (B8.19) sur les mêmes ratios (détachés) et les mêmes avantages, dont l'écart à `parts["policy_loss"]` entre dans le journal ;
  - **`reward_fn(obs, actions, next_obs)`**, quand elle est donnée, remplace les récompenses du rollout par les siennes (tableau `(N,)` pour un pas) avant GAE : le RLHF de la partie C s'en servira ; `"mean_return"` reste calculé avec les **vraies** récompenses ;
  - **`logs`** : un dictionnaire de listes, une valeur par itération, aux clés de `wbr.LOG_KEYS` (`"step"`, pas d'environnement joués jusque-là, toutes copies comprises ; `"iteration"` ; `"lr"` ; `"policy_loss"`, `"value_loss"`, `"entropy"`, `"approx_kl"`, `"clipfrac"`, moyennes sur les mini-batches de l'itération ; `"explained_variance"` ; `"episodes"`, le nombre d'épisodes finis pendant le rollout ; `"mean_return"`, leur retour moyen, `nan` s'il n'y en a pas) et `"surrogate_gap"` (le plus grand écart de l'itération entre ta `clipped_surrogate` et la loss torch).

La vérification essaie `ActorCritic20` ; puis `rollout_20` avec un agent fixe de la référence, deux rollouts de 10 pas sur deux copies coupées à 12 pas : elle rejoue l'environnement avec **tes** actions et compare chaque champ du batch ; puis `ppo_loss_20` sur un mini-batch de 16 contre la référence ; enfin une boucle de deux itérations, qui observe tes appels à `compute_gae` et à `clipped_surrogate`. La cellule suivante entraîne ton PPO sur `STEPS_20` pas (50 000 en mode rapide, environ 25 s), trace le retour de chaque rollout et évalue la politique **déterministe** (l'action la plus probable) sur 100 épisodes ; elle attend que le retour monte et que l'évaluation dépasse `THRESHOLD_20`, puis enregistre ton agent (`AGENT_KEY_20`, à ton nom) pour la suite du chapitre.

Dans tes notes : combien de pas de reset ton rollout a-t-il écartés sur 50 000 pas, et que se serait-il passé en les gardant (🕰️ Gymnasium 1.x) ? Où ta boucle amorce-t-elle une limite de temps, et que fait `ppo.py` de CleanRL à la place (208.4) ? Pourquoi normaliser les avantages par mini-batch ? Combien de pas d'Adam ton PPO a-t-il faits, et combien de transitions chacun a-t-il servi ?

In [ ]:
SETTINGS_20 = dict(wbr.PPO_SETTINGS)      # the settings of the reference PPO (printed below)
STEPS_20 = wbr.ppo_steps(FAST_MODE)         # 50 000 environment steps (all copies) in fast mode, 100 000 in full mode
SEED_20 = 20
THRESHOLD_20 = 170               # the deterministic evaluation that the check asks (6 seeds of the reference)
AGENT_KEY_20 = wbr.agent_key(SETTINGS_20, source=SOURCE, fast=FAST_MODE, seed=SEED_20, total_steps=STEPS_20)


def load_agent_20(verbose=True):
    """(agent, source) for the rest of the chapter: the agent trained by Ex B8.20 in this session (AGENT_20), else the
    one an earlier session saved under AGENT_KEY_20, else the reference PPO agent of wbr (an explicit fallback, said).
    The solutions notebook never reads the learner's agent: its key has the source "solution"."""
    if globals().get("AGENT_20") is not None:
        return AGENT_20, SOURCE
    try:
        loaded = wbr.load_agent(AGENT_KEY_20, build=ActorCritic20)
    except NotImplementedError:
        loaded = None
    if loaded is not None:
        if verbose:
            print(f"ℹ️ agent de B8.20 relu ({loaded[1]['path'].name}, enregistré le {loaded[1]['saved_at']})")
        return loaded[0], SOURCE
    agent, _ = wbr.reference_agent(FAST_MODE, verbose=verbose)
    if verbose:
        print("ℹ️ pas d'agent de B8.20 enregistré : repli sur l'agent PPO de référence (wbr.reference_agent)")
    return agent, "reference"


print(SETTINGS_20)

In [ ]:
class ActorCritic20(nn.Module):
    """The agent: an actor (4 -> hidden -> hidden -> 2 logits, tanh) and a critic (4 -> hidden -> hidden -> 1, tanh),
    two separate networks in the attributes `actor` and `critic`. forward(obs) returns the logits (N, 2);
    value(obs) returns V(obs), of shape (N,)."""

    def __init__(self, hidden=64):
        super().__init__()
        raise NotImplementedError("ActorCritic20.__init__")

    def forward(self, obs):
        raise NotImplementedError("ActorCritic20.forward")

    def value(self, obs):
        raise NotImplementedError("ActorCritic20.value")


def rollout_20(agent, envs, obs, just_reset, n_steps, generator):
    """One rollout of n_steps steps in each copy of envs (the contract of the statement). Returns (batch, obs,
    just_reset): batch a dict of NumPy arrays of shape (n_steps, N, ...) "obs", "actions", "log_probs", "values",
    "rewards", "terminated", "truncated", "next_obs", "next_values", "valid"; obs and just_reset for the next rollout."""
    raise NotImplementedError("rollout_20")


def ppo_loss_20(agent, obs, actions, old_log_probs, advantages, returns, settings=SETTINGS_20):
    """The PPO loss of a mini-batch of tensors (the contract of the statement): (loss, parts), loss a tensor with its
    graph, parts a dict of floats "policy_loss", "value_loss", "entropy", "approx_kl", "clipfrac"."""
    raise NotImplementedError("ppo_loss_20")


def train_ppo_20(seed, total_steps, settings=SETTINGS_20, reward_fn=None):
    """PPO on settings["n_envs"] copies of CartPole-v1 (the contract of the statement). Returns (agent, logs), logs a
    dict of lists (one value per iteration) with the keys of wbr.LOG_KEYS and "surrogate_gap"."""
    raise NotImplementedError("train_ppo_20")

In [ ]:
ok_20 = False
with checked("B8.20"):
    torch.manual_seed(0)
    agent_20 = ActorCritic20()
    obs_20 = torch.as_tensor(np.random.default_rng(20).normal(0, 0.1, (5, 4)), dtype=torch.float32)
    logits_20, value_20 = agent_20(obs_20), agent_20.value(obs_20)
    has_parts_20 = isinstance(getattr(agent_20, "actor", None), nn.Module) and isinstance(getattr(agent_20, "critic", None),
                                                                                         nn.Module)
    verdict("B8.20", tuple(logits_20.shape) == (5, 2) and tuple(value_20.shape) == (5,) and has_parts_20,
            f"ActorCritic20 : acteur et critique séparés, {sum(p.numel() for p in agent_20.parameters())} paramètres ; "
            "forward rend (5, 2), value rend (5,).",
            f"ActorCritic20 : attendu des attributs actor et critic, forward de forme (5, 2) et value de forme (5,), reçu "
            f"{tuple(logits_20.shape)} et {tuple(value_20.shape)}" + ("" if has_parts_20 else ", sans actor ou critic") + ".")

rollout_ok_20 = False
with checked("B8.20"):
    torch.manual_seed(20)
    ref_agent_20 = wbr.ActorCritic(64)                    # a fixed agent of the reference, so that only rollout_20 is tested
    envs_20 = wbr.make_vec(2, seed=5, max_episode_steps=12)
    gen_20 = torch.Generator().manual_seed(20)
    out_a_20 = rollout_20(ref_agent_20, envs_20, envs_20.first_obs.copy(), np.zeros(2, bool), 10, gen_20)
    batch_a_20, obs_a_20, reset_a_20 = pair("B8.20", "rollout_20", out_a_20, names=("batch", "obs", "just_reset"))
    out_b_20 = rollout_20(ref_agent_20, envs_20, obs_a_20, reset_a_20, 10, gen_20)
    batch_b_20, _, _ = pair("B8.20", "rollout_20", out_b_20, names=("batch", "obs", "just_reset"))
    keys_20 = ("obs", "actions", "log_probs", "values", "rewards", "terminated", "truncated", "next_obs", "next_values",
               "valid")
    missing_20 = [k for k in keys_20 if not isinstance(batch_a_20, dict) or k not in batch_a_20]
    if missing_20:
        print(f"❌ Ex B8.20 : le batch de rollout_20 doit avoir les clés {list(keys_20)} ; il manque {missing_20}.")
        raise StopCheck
    shapes_20 = {k: np.shape(batch_a_20[k]) for k in keys_20}
    want_shapes_20 = {k: ((10, 2, 4) if k in ("obs", "next_obs") else (10, 2)) for k in keys_20}
    if shapes_20 != want_shapes_20:
        print(f"❌ Ex B8.20 : formes du batch : attendu {want_shapes_20}, reçu {shapes_20}.")
        raise StopCheck
    replay_20 = wbr.make_vec(2, seed=5, max_episode_steps=12)          # the same copies, played with YOUR actions
    actions_20 = np.concatenate([batch_a_20["actions"], batch_b_20["actions"]]).astype(np.int64)
    expect_20 = {k: [] for k in ("obs", "rewards", "terminated", "truncated", "next_obs", "valid")}
    o_20, jr_20 = replay_20.first_obs, np.zeros(2, bool)
    for t_20 in range(20):
        n_20, r_20, te_20, tr_20, _ = replay_20.step(actions_20[t_20])
        for k_20, v_20 in zip(expect_20, (o_20, r_20, te_20, tr_20, n_20, ~jr_20)):
            expect_20[k_20].append(np.array(v_20, copy=True))
        jr_20, o_20 = te_20 | tr_20, n_20
    both_20 = {k: np.concatenate([np.asarray(batch_a_20[k]), np.asarray(batch_b_20[k])]) for k in keys_20}
    problems_20 = []
    for k_20, v_20 in expect_20.items():
        if not np.allclose(both_20[k_20].astype(float), np.stack(v_20).astype(float)):
            problems_20.append(k_20)
    with torch.no_grad():
        x_20 = torch.as_tensor(both_20["obs"].reshape(-1, 4), dtype=torch.float32)
        lp_want_20 = torch.log_softmax(ref_agent_20(x_20), 1).gather(
            1, torch.as_tensor(actions_20.reshape(-1, 1))).squeeze(1).numpy().reshape(20, 2)
        v_want_20 = ref_agent_20.value(x_20).numpy().reshape(20, 2)
        nv_want_20 = ref_agent_20.value(torch.as_tensor(both_20["next_obs"].reshape(-1, 4), dtype=torch.float32)
                                        ).numpy().reshape(20, 2)
    for k_20, want_20 in (("log_probs", lp_want_20), ("values", v_want_20), ("next_values", nv_want_20)):
        if not np.allclose(both_20[k_20].astype(float), want_20, atol=1e-5):
            problems_20.append(k_20)
    ends_20 = int((np.stack(expect_20["terminated"]) | np.stack(expect_20["truncated"])).sum())
    if not problems_20:
        rollout_ok_20 = True
        print(f"✅ Ex B8.20 : deux rollouts de 10 pas sur 2 copies coupées à 12 pas ({ends_20} fins d'épisode) : tout "
              "correspond à l'environnement rejoué avec tes actions ; les pas de reset sont écartés (valid), "
              "next_values est V de l'observation qui suit chaque pas, et le second rollout reprend où le premier s'arrête.")
    else:
        hints_20 = {"valid": "valid doit valoir False exactement au pas qui suit une fin d'épisode dans la même copie "
                             "(ce pas ne fait que le reset, 208.5), y compris d'un rollout au suivant (just_reset)",
                    "next_values": "next_values[t] est V de l'observation que rend le pas t (la vraie dernière quand "
                                   "l'épisode finit), pas la valeur du pas suivant du rollout",
                    "obs": "obs[t] est l'observation AVANT le pas t ; le second rollout repart de l'observation rendue "
                           "par le premier",
                    "log_probs": "log_probs est le log de la probabilité de l'action jouée, sous la politique de l'agent",
                    "values": "values est V(obs[t]) du critique de l'agent"}
        for k_20 in problems_20:
            print(f"❌ Ex B8.20 : rollout_20, « {k_20} » ne correspond pas à l'environnement rejoué avec tes actions"
                  + (f" : {hints_20[k_20]}." if k_20 in hints_20 else "."))

loss_ok_20 = False
with checked("B8.20"):
    rng_20 = np.random.default_rng(200)
    mb_20 = {"obs": torch.as_tensor(rng_20.normal(0, 0.5, (16, 4)), dtype=torch.float32),
             "actions": torch.as_tensor(rng_20.integers(0, 2, 16)),
             "advantages": torch.as_tensor(rng_20.normal(size=16), dtype=torch.float32),
             "returns": torch.as_tensor(rng_20.normal(5, 2, 16), dtype=torch.float32)}
    with torch.no_grad():                                # old log-probabilities a little away from the current ones
        mb_20["old_log_probs"] = torch.log_softmax(ref_agent_20(mb_20["obs"]), 1).gather(
            1, mb_20["actions"][:, None]).squeeze(1) + torch.as_tensor(rng_20.normal(0, 0.3, 16), dtype=torch.float32)
    args_20 = (mb_20["obs"], mb_20["actions"], mb_20["old_log_probs"], mb_20["advantages"], mb_20["returns"])
    loss_20, parts_20 = pair("B8.20", "ppo_loss_20", ppo_loss_20(ref_agent_20, *args_20, SETTINGS_20),
                             names=("loss", "parts"))
    want_loss_20, want_parts_20 = wbr.ppo_loss(ref_agent_20, *args_20, SETTINGS_20)
    good_20 = [same("B8.20", "ppo_loss_20 sur un mini-batch de 16 (la loss)", loss_20.detach(), want_loss_20.detach(),
                    rtol=1e-5, atol=1e-6)]
    for k_20, v_20 in want_parts_20.items():
        got_20 = parts_20.get(k_20) if isinstance(parts_20, dict) else None
        good_20.append(got_20 is not None and abs(float(got_20) - v_20) <= 1e-5 * max(1.0, abs(v_20)))
        if not good_20[-1]:
            print(f"❌ Ex B8.20 : ppo_loss_20, parts[{k_20!r}] : attendu {v_20:.6g}, reçu {got_20!r}.")
    if not good_20[0] and all(good_20[1:]):
        print("❌ Ex B8.20 : les cinq parties sont justes, mais la loss ne les combine pas comme le contrat : "
              "policy_loss + vf_coef × value_loss − ent_coef × entropy (signes et coefficients de SETTINGS_20).")
    ref_agent_20.zero_grad()
    loss_20.backward()
    reached_20 = {name: any(p.grad is not None and float(p.grad.abs().sum()) > 0 for p in m.parameters())
                  for name, m in (("acteur", ref_agent_20.actor), ("critique", ref_agent_20.critic))}
    good_20.append(all(reached_20.values()))
    verdict("B8.20", good_20[-1], "le gradient de la loss atteint l'acteur et le critique.",
            f"le gradient de la loss n'atteint pas {', '.join(n for n, ok in reached_20.items() if not ok)} : un .detach() "
            "ou un .item() de trop ?")
    loss_ok_20 = all(good_20)

with checked("B8.20"):
    if not (rollout_ok_20 and loss_ok_20):
        print("⏳ Ex B8.20 : la boucle se vérifie une fois rollout_20 et ppo_loss_20 justes.")
    elif not (written(lambda: mylearn.deep_rl.compute_gae([1.0], [0.0], [0.0], [True], [False]))
              and written(lambda: mylearn.deep_rl.clipped_surrogate([1.0], [1.0]))):
        print("⏳ Ex B8.20 : ta boucle se sert de compute_gae et de clipped_surrogate de ta librairie (B8.18, B8.19), pas "
              "encore écrites.")
    else:
        seen_20 = {"gae": [], "surrogate": 0}
        real_gae_20, real_clip_20 = mylearn.deep_rl.compute_gae, mylearn.deep_rl.clipped_surrogate

        def spy_gae_20(*args, **kwargs):
            out = real_gae_20(*args, **kwargs)
            bound = inspect.signature(real_gae_20).bind(*args, **kwargs).arguments
            seen_20["gae"].append((tuple(np.shape(bound.get(k)) for k in ("rewards", "values", "next_values",
                                                                           "terminated", "truncated")), args, kwargs))
            return out

        def spy_clip_20(*args, **kwargs):
            seen_20["surrogate"] += 1
            return real_clip_20(*args, **kwargs)

        mylearn.deep_rl.compute_gae, mylearn.deep_rl.clipped_surrogate = spy_gae_20, spy_clip_20
        try:
            short_20 = {**SETTINGS_20, "n_epochs": 2}
            per_iter_20 = short_20["n_envs"] * short_20["n_steps"]
            agent_s_20, logs_s_20 = pair("B8.20", "train_ppo_20", train_ppo_20(3, 2 * per_iter_20, short_20),
                                         names=("agent", "logs"))
        finally:
            mylearn.deep_rl.compute_gae, mylearn.deep_rl.clipped_surrogate = real_gae_20, real_clip_20
        want_keys_20 = set(wbr.LOG_KEYS) | {"surrogate_gap"}
        good_20 = []
        good_20.append(isinstance(logs_s_20, dict) and want_keys_20 <= set(logs_s_20)
                       and all(len(logs_s_20[k]) == 2 for k in want_keys_20))
        verdict("B8.20", good_20[-1], "les journaux : une valeur par itération pour chaque clé de wbr.LOG_KEYS et surrogate_gap.",
                f"logs doit avoir les clés {sorted(want_keys_20)}, une valeur par itération (2 ici), reçu "
                + (str({k: len(v) for k, v in logs_s_20.items()}) if isinstance(logs_s_20, dict) else type(logs_s_20).__name__) + ".")
        shape_20 = (short_20["n_steps"], short_20["n_envs"])
        good_20.append(len(seen_20["gae"]) == 2 and all(s == (shape_20,) * 5 for s, _, _ in seen_20["gae"]))
        verdict("B8.20", good_20[-1], f"compute_gae de ta librairie : appelée une fois par itération sur des tableaux {shape_20}.",
                f"attendu un appel de mylearn.deep_rl.compute_gae par itération (2) sur des tableaux {shape_20} (le temps "
                f"d'abord), reçu {len(seen_20['gae'])} appels" + (f" de formes {seen_20['gae'][0][0]}" if seen_20["gae"] else "")
                + " ; l'appelles-tu par ce nom (la vérification l'observe) ?")
        n_mb_20 = 2 * short_20["n_epochs"] * short_20["n_minibatches"]
        good_20.append(seen_20["surrogate"] >= n_mb_20)
        verdict("B8.20", good_20[-1], f"clipped_surrogate de ta librairie : appelée à chaque mini-batch ({seen_20['surrogate']} fois).",
                f"attendu au moins {n_mb_20} appels de mylearn.deep_rl.clipped_surrogate (un par mini-batch), reçu "
                f"{seen_20['surrogate']}.")
        if good_20[0]:
            gap_20 = max(logs_s_20["surrogate_gap"])
            good_20.append(gap_20 <= 1e-5)
            verdict("B8.20", good_20[-1], f"ta clipped_surrogate et la loss torch coïncident (écart {gap_20:.1e}).",
                    f"ta clipped_surrogate et la loss torch du même mini-batch diffèrent de {gap_20:.2g} : leur passes-tu "
                    "les mêmes ratios et les mêmes avantages (normalisés) ?")
        ok_20 = all(good_20)

In [ ]:
# Training: your PPO, STEPS_20 steps on 4 copies, then the deterministic policy on 100 episodes; the agent is saved
if not ok_20:
    print("⏳ Ex B8.20 : l'entraînement attend que la vérification ci-dessus passe.")
else:
    with checked("B8.20"):
        start_20 = time.perf_counter()
        AGENT_20, LOGS_20 = train_ppo_20(SEED_20, STEPS_20)
        seconds_20 = time.perf_counter() - start_20
        eval_20 = wbr.evaluate(wbr.greedy_logits(AGENT_20), 100, EVAL_SEED)
        steps_20, returns_20 = np.asarray(LOGS_20["step"]), np.asarray(LOGS_20["mean_return"], dtype=float)
        fig_20, ax_20 = plt.subplots(figsize=(7, 3))
        ax_20.plot(steps_20, returns_20, marker=".", label="retour moyen des épisodes finis de chaque rollout")
        ax_20.axhline(eval_20["mean_return"], color="k", ls="--", lw=1, label="politique déterministe, 100 épisodes")
        ax_20.set(xlabel="pas d'environnement (4 copies)", ylabel="retour", title=f"Ton PPO, graine {SEED_20}")
        ax_20.legend(fontsize=8)
        plt.show()
        print(f"{STEPS_20} pas en {seconds_20:.0f} s ({len(steps_20)} itérations, {int(np.sum(LOGS_20['episodes']))} "
              "épisodes finis) · politique déterministe : "
              f"{fr(eval_20['mean_return'], 1)} sur 100 épisodes (écart-type {fr(eval_20['std_return'], 1)}) · écart "
              f"maximal entre ta clipped_surrogate et la loss torch : {max(LOGS_20['surrogate_gap']):.1e}")
        finite_20 = returns_20[np.isfinite(returns_20)]          # an iteration where no episode ends has nan
        early_20, late_20 = float(finite_20[:5].mean()), float(finite_20[-5:].mean())
        verdict("B8.20", late_20 >= 2 * early_20,
                f"le retour monte : {fr(early_20, 1)} sur les 5 premières itérations, {fr(late_20, 1)} sur les 5 dernières.",
                f"le retour ne monte pas assez : {fr(early_20, 1)} sur les 5 premières itérations, {fr(late_20, 1)} sur "
                "les 5 dernières.")
        verdict("B8.20", eval_20["mean_return"] >= THRESHOLD_20,
                f"la politique déterministe dépasse {THRESHOLD_20:g} (la référence, sur 6 graines : 500 ; 500 ; 500 ; 284 ; 500 ; 500).",
                f"la politique déterministe tient {fr(eval_20['mean_return'], 1)} pas : moins que les {THRESHOLD_20:g} "
                f"attendus (la référence, sur 6 graines : 500 ; 500 ; 500 ; 284 ; 500 ; 500).")
        wbr.delete_agent(AGENT_KEY_20)
        print(f"agent enregistré pour la suite du chapitre (source {SOURCE!r}) : "
              f"{wbr.save_agent(AGENT_KEY_20, AGENT_20, LOGS_20, extra={'eval': eval_20['mean_return']}).name}")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une cible DQN et Double DQN, en distinguant une vraie fin d'une limite de temps ?
2. Sais-tu dériver le gradient du log d'une politique softmax, et dire pourquoi une baseline ne biaise pas le gradient ?
3. Sais-tu calculer GAE à rebours et l'objectif clippé de PPO avec son gradient, cas par cas ?
4. Sais-tu calculer une loss de Bradley-Terry et une loss DPO, et dire ce que règle $\beta$ ?

**Pour aller plus loin** : l'article de Schulman et coll. (📄 B8.11) et les « 37 détails » de Huang et coll., cités dans la fiche. B8 est le dernier chapitre bonus.

*Les exercices B8.21 à B8.28 (la fin de la politique, les préférences et le défi) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B8` pour obtenir le notebook complet.*